# Python程序设计（项目实战） · 工单P7《智能运维平台综合项目》
**第9周 | 2课时(80分钟) | 理实一体**  | 教师参考版

> **剧情**：方总监验收完 P6，把服务器钥匙拍在桌上——"模型不错，但数据还躺在十二个文件夹里。把它串成一个**不睡觉的运维平台**。"
> **载体**：重庆两江智造集团·数字化转型办公室 ｜ 地点：双创中心302实训室 ｜ 提交命名：`学号_姓名_P7`

| 层 | 目标 | 任务 | 达标标准 |
|---|---|---|---|
| **D 学徒** | 跑通平台 | `bash deploy/run.sh` → 四个 API 各调一次 | 4个API全部返回200，能说出 live→cache 两次的差异 |
| **C 操作员** | 补查询 | TODO-P7-1 关系台账 / P7-2 时序窗口 / P7-4 向量检索 | `verify` **五库段全绿** |
| **B 组长** | 通智能体 | TODO-P7-3 图影响路径 / P7-5 ReAct 循环；问题换 FAN-02 再测一轮 | `verify` **ALL PASS** |
| **A 师傅** | 上线级 | 嵌入召回对比 + `ab -n 1000 -c 50` 压测调 workers + docker compose 部署 | 写出**选型 + 压测对比报告**（召回率/QPS 前后对照） |

> 每4周按工单成绩动态升降层（连续2次≥B升一层）
> 评分：功能40% + **技术质量25%（决策矩阵与压测留痕）** + 代码规范20% + 工程素养15%（D≥60 / C≥70 / B≥80 / A≥90）
> 思政观测点：**有据可查**（智能体每步留痕）+ **全绿才上线**（端到端断言不红才签字）


## 一、课前资讯（Predict·预测）
> **本课教材**：《Python程序设计项目化教程（活页式）》项目6 拓展 ｜ **预习知识点**：多模存储 / 异步服务 / 智能体
> **预习快问**：台账、时序、拓扑、案例、缓存这五类数据分别该放哪种库？为什么不能全塞进一个 SQLite？
> **上周抽查**：P6 的 F1≥0.80 —— 随机2人口述/白板，答对记过程分

**本课知识四要素**（与授课PPT同源，下面每一格代码都对着一条）：
- **B01 多模存储各就各位**：五类库=五种货架——SQLite台账 / 列式时序 / KV秒取 / 向量语义 / 图关系网。误区：一把锤子敲所有钉。
- **B02 Litestar 全异步 + Granian(Rust)**：误区**「异步当万能加速」——CPU 密集照样慢，红利在 IO 等待**（Starter 21 用两个基准端点现场证明）。
- **B03 ReAct 智能体**：想→动手→看线索→结案，取证链完整才可信。误区：让模型一步出结论。
- **B04 缓存三问**：什么值得缓存？TTL 多长？过期谁负责？（穿透→空值也要缓存；雪崩→TTL 加抖动）
- **B05 端到端验证思维**：verify 断言全绿才签字放行——不靠人眼靠断言。

## 二、PRIMM五步法（先读后写）
1. **Predict 预测**——每格运行前先猜：会打印什么？断言会不会红？为什么？
2. **Run 运行**——严格按顺序跑（五库一格一格播种，跳格会 KeyError）；
3. **Investigate 研究**——读懂脚手架：函数签名、必须返回的键、`坑：`注释；
4. **Modify 修改**——补齐 5 个 TODO（P7-1 / P7-2 / P7-4 = C层，P7-3 / P7-5 = B层，P7-A = A层）；
5. **Make 创造**——独立跑到 `ALL PASS ✅`，并把技术决策矩阵填成自己的话。


## ⚠ 课堂环境说明：原厂引擎 → 离线等价实现（先把边界讲透，免得"跑通了但不是工单那套"）

实训机（conda 环境 `pp03`）**没有装** litestar / granian / duckdb / kuzu / chromadb / redis / streamlit / paho-mqtt。
下面第 1 格会做一次 **"引擎体检"** 并如实打印缺哪些——我们不假装装上，**也不伪造任何输出数字**。
处理原则：**接口契约与验收口径一行不改**，只把引擎换成本机白名单（标准库 + numpy/pandas/scikit-learn/sqlalchemy/joblib）的等价实现。

| 工单原厂 | 本课离线等价 | 契约一致性 | 换回原厂要动哪一处 |
|---|---|---|---|
| SQLite(WAL) 台账 | SQLAlchemy 2.0 + SQLite，**真开 WAL**（`journal_mode` 实测打印） | 完全一致 | 不用换，本来就是 SQLite |
| DuckDB 窗口/ASOF | pandas 列式 + `pd.merge_asof(direction="backward")` | 原厂 SQL 逐行写在注释里对照 | 只换 `window_feats()` 函数体 |
| Kùzu 嵌入式图 | 邻接表 + BFS 变长跳 `follow(dev, rel, hops=(1,3))` | = `[:上游供电*1..3]`，docstring 保留原 Cypher | 只替换 `follow()` |
| Chroma 向量库 | 中文哈希嵌入（字×1 + bigram×2）+ numpy 余弦 + `.npz` 持久化 | 含 `where=` 等价的 **device 元数据过滤** | 只换 `embed_hash()`（如 bge-small-zh） |
| Redis KV | 标准库 json 落盘 TTL-KV + **可注入假时钟**（过期当场可见，不用 sleep） | `get/put/incr/ttl` 同签名 | 设 `P77_USE_REDIS=1` 且有服务端 |
| Litestar + Granian | `http.server.ThreadingHTTPServer` + 路由表分发 | **4 条路由/字段/状态码完全一致**，429/404/400 都在 | 换 ASGI app，handler 不动 |
| `deploy/run.sh` + `verify.sh` | notebook 内生成同名文件 + `verify()` 函数 | 同名同职责 | 直接调用 |

**本课确实做不到的（诚实清单，答辩不吹）**：
1. **bge-small-zh 真实语义嵌入**——权重要联网下载，本机不可测 → Starter 20 **不给它编召回数字**，只做三套离线方案对照；
2. **真·多 worker 进程压测**——无 granian，Starter 21 用「单线程 vs 多线程 + IO端点 vs CPU端点」复刻同一个结论，
   并明确标注：同进程 stdlib 客户端受 GIL 压制，**压测工具也是被测系统的一部分**；
3. **容器真部署**——`docker compose config` 只在守护进程可用时才真跑，否则如实标注『未执行』；
4. **streamlit 大屏 / MQTT 设备接入**——工单 P7 的栈里没有它们（属 P3/P4 范围），本 notebook 不涉及，不做假演示；
5. 工单要求的**"截图留痕"**在 notebook 里改成把两次响应体逐字落盘 `w09_scratch/reports/`——**留痕比截图更可查**。


## 三、任务要求（Make）

**实施步骤（对应工单"五步上手"）**
1. **配置与五库播种**：关系台账 → 时序明细 → 拓扑图 → 案例向量 → KV 缓存（一库一格，每格可单独运行看效果）
2. **三个查询函数**（C层）：`device_profile` P7-1 / `window_feats` P7-2 / `search` P7-4
3. **两个链路函数**（B层）：`impact_path` P7-3 / `investigate` P7-5（ReAct 五步取证）
4. **服务与总检**：4 条路由 + 限流 429 + `verify()` 端到端断言
5. **交付**（A层）：`deploy/run.sh` + `Dockerfile` + `docker-compose.yml` + 技术决策矩阵 + 选型/压测对比报告

**API 验收口径**（原厂 `granian 127.0.0.1:8900`；本课同契约、随机空闲端口起停，不撞课堂端口）

| 接口 | 期望返回 | 观察点 |
|---|---|---|
| `GET /health` | `{"ok":true,"kv":"…","model_f1":…}` | KV 用的是 redis 还是离线兜底 |
| `GET /api/device/PUMP-03` | 台账 + 测点 + 告警 + `ts_feats` | 连刷两次：`source` 由 `live` 变 `cache`，数据**逐字相同** |
| `POST /api/predict` `{"device":"PUMP-03"}` | `proba_wear` + `pred.label` | 第一次 `MISS`，第二次 `HIT` |
| `POST /api/agent/ask` `{"question":"…"}` | 五步 `trace` + `conclusion.suggest_action` | 每步含 thought/act/args/obs（有据可查） |

**技术决策矩阵（必填，答辩必问）**——本 notebook 每格都会打印实测值，**抄自己的数字，不要抄模板**：

| 场景 | 候选A | 候选B | 你的选择与理由（要引用实测数字） |
|---|---|---|---|
| 设备台账 | SQLite(WAL) | MongoDB | Starter 2：`journal_mode` 实测=? |
| 分钟级窗口特征 | DuckDB(列式) | SQLite逐行 | Starter 5：列式 ?ms vs 逐行 ?ms = ?× |
| 拓扑影响路径 | Kùzu嵌入式 | Neo4j服务 | Starter 8：影响面覆盖率断言=?% |
| 故障案例检索 | Chroma+嵌入 | 全文LIKE | Starter 20：三套方案 recall@1 = ? / ? / ? |
| Web服务 | Litestar+Granian | http.server | Starter 21：IO 型 ?× / CPU 型 ?× |

**验收标准**：D=`4个API全200` ｜ C=`五库段全绿` ｜ B=`verify ALL PASS（含 FAN-02 复测）` ｜ A=`选型+压测对比报告齐套`


In [5]:
# -*- coding: utf-8 -*-
# --- 环境与数据准备(勿改, 运行即可) ---
import json
import os
import shutil
import sqlite3
import subprocess
import sys
import time
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

SEED = 7                       # 全局随机种子：本机与机房跑出的结果完全一致
BASE = os.path.abspath(os.getcwd())
SCRATCH = os.path.join(BASE, "w09_scratch")     # 所有库/缓存/报告都写在这里
shutil.rmtree(SCRATCH, ignore_errors=True)       # 每次从干净状态开始（可重复运行）
for _sub in ("data", "deploy", "reports"):
    os.makedirs(os.path.join(SCRATCH, _sub), exist_ok=True)

print("工作目录 :", BASE)
print("实验目录 :", SCRATCH, "(课程包根目录不落任何文件)")
print("Python   :", sys.version.split()[0])

# 离线课堂"引擎体检"：工单点名的重型引擎是否可用，只如实打印，不使用
CLOUD_ONLY = ["litestar", "granian", "duckdb", "kuzu", "chromadb", "redis", "streamlit", "paho.mqtt.client"]
print("\n[引擎体检] 工单P7原厂组件 → 本机状态")
for m in CLOUD_ONLY:
    try:
        __import__(m)
        state = "本机可导入"
    except ImportError:
        state = "本机未装"
    print(f"  - {m:<16} {state} → 本课**不使用**，改用离线白名单等价实现(见下一格)")

import numpy
import pandas
import sklearn
import sqlalchemy
print("\n[白名单] numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scikit-learn", sklearn.__version__, "| SQLAlchemy", sqlalchemy.__version__)


工作目录 : /Users/joelgong/PycharmProjects/ai-course-teaching-resources/03-Python程序设计-项目实战/Jupyter实训
实验目录 : /Users/joelgong/PycharmProjects/ai-course-teaching-resources/03-Python程序设计-项目实战/Jupyter实训/w09_scratch (课程包根目录不落任何文件)
Python   : 3.12.11

[引擎体检] 工单P7原厂组件 → 本机状态
  - litestar         本机可导入 → 本课**不使用**，改用离线白名单等价实现(见下一格)
  - granian          本机可导入 → 本课**不使用**，改用离线白名单等价实现(见下一格)
  - duckdb           本机可导入 → 本课**不使用**，改用离线白名单等价实现(见下一格)
  - kuzu             本机可导入 → 本课**不使用**，改用离线白名单等价实现(见下一格)
  - chromadb         本机可导入 → 本课**不使用**，改用离线白名单等价实现(见下一格)
  - redis            本机可导入 → 本课**不使用**，改用离线白名单等价实现(见下一格)
  - streamlit        本机可导入 → 本课**不使用**，改用离线白名单等价实现(见下一格)
  - paho.mqtt.client 本机可导入 → 本课**不使用**，改用离线白名单等价实现(见下一格)

[白名单] numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | SQLAlchemy 2.1.3


### Starter 1：平台配置层（一库一文件 + 开关约定）
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

三条工程纪律（答辩必问）：**路径全部集中在 config 层**（业务代码不写死绝对路径）／
**任何要联网或要守护进程的组件都有本地兜底开关**／**运行产物只写 `w09_scratch/`，课程包根目录零污染**。
端口写 `0` 让内核挑空闲端口——课堂 40 台机器同时起服务也不会撞车。

In [7]:
# -*- coding: utf-8 -*-
"""平台配置（对应 starter 的 config.py）：一库一文件 + 端口/开关约定

三条工程纪律（答辩必问）：
  1. 路径全部集中在 config 层，业务代码不写死绝对路径；
  2. 机房离线可跑：任何"要联网/要守护进程"的组件都有本地兜底实现；
  3. 一库一文件：五类存储互不依赖，可单独重建（对应 `python -m stores.xxx`）。
"""

DATA = os.path.join(SCRATCH, "data")
MODELS = os.path.join(DATA, "models")
REPORTS = os.path.join(SCRATCH, "reports")
os.makedirs(MODELS, exist_ok=True)

SQLITE_FILE = os.path.join(DATA, "smart_ops.db")        # 台账(关系) + 原始采样(对照)
TS_PKL = os.path.join(DATA, "vib_columnar.pkl")          # 时序列存快照（DuckDB 的离线替身）
GRAPH_JSON = os.path.join(DATA, "topology.json")         # 拓扑图（Kùzu 的离线替身）
VEC_NPZ = os.path.join(DATA, "fault_cases.npz")          # 向量索引（Chroma 的离线替身）
KV_FILE = os.path.join(DATA, "kv_cache.json")            # KV 缓存（Redis 的离线替身）
CLF_FILE = os.path.join(MODELS, "p77_clf.joblib")        # P6 训练好的诊断模型

SQLITE_URL = f"sqlite:///{SQLITE_FILE}"
SERVICE_HOST = "127.0.0.1"
CACHE_TTL_DEV = 10        # 秒：大屏轮询类，短 TTL
CACHE_TTL_PRED = 10       # 秒：模型推理结果
CACHE_JITTER = 5          # 秒：TTL 抖动，防"雪崩"（同刻集体过期）
RATE_LIMIT_PER_HOUR = 60  # 智能体接口限流
print("配置就绪 →", os.path.relpath(DATA, BASE))


配置就绪 → w09_scratch/data


### Starter 2：存储引擎① 关系台账（SQLAlchemy 2.0 + SQLite **WAL**）
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

三张表就是工厂里最常见的「主表 + 从表」：`devices 1─N sensor_points`、`devices 1─N alerts`。
两个必须点出来的细节：
- `relationship()` 让「一台泵有哪些测点/告警」不用手写 JOIN——这是 ORM 给运维的第一份礼物；
- 连接事件里 `PRAGMA journal_mode=WAL`：默认 `delete` 模式下**写会阻塞读**，大屏一刷新就 `database is locked`。
本格把 `journal_mode` **实测打印并 assert 等于 `wal`**——决策矩阵第一行的理由必须是你自己查到的。

In [9]:
# -*- coding: utf-8 -*-
"""存储引擎① 关系型：SQLAlchemy 2.0 + SQLite(WAL) —— 设备台账/测点/告警单

对应 starter `stores/relational.py`。三张表就是工厂里最常见的"主表+从表"：
  devices 1─N sensor_points   （一台泵有多个测点）
  devices 1─N alerts          （一台泵有多条告警）
WAL 日志模式让"大屏在读、采集在写"互不阻塞——这是工单决策矩阵里
"设备台账选 SQLite(WAL) 而不是 MongoDB"的落地前提。
"""
from sqlalchemy import (JSON, Column, DateTime, Float, ForeignKey, Integer,
                        String, create_engine, event, select, text)
from sqlalchemy.orm import DeclarativeBase, Session, relationship


class Base(DeclarativeBase):
    pass


class Device(Base):
    """设备台账（档案主表）  数据字典: code 唯一编号 / line_code 所属产线(与图库同源)"""
    __tablename__ = "devices"
    id = Column(Integer, primary_key=True)
    code = Column(String(16), unique=True)
    name = Column(String(64))
    line_code = Column(String(16))
    installed = Column(DateTime, default=datetime(2024, 3, 1))
    points = relationship("SensorPoint", back_populates="device")
    alerts = relationship("Alert", back_populates="device")


class SensorPoint(Base):
    """测点表  tag: vib_rms / pressure / temp"""
    __tablename__ = "sensor_points"
    id = Column(Integer, primary_key=True)
    device_id = Column(ForeignKey("devices.id"))
    tag = Column(String(32))
    unit = Column(String(8), default="mm/s")
    device = relationship("Device", back_populates="points")


class Alert(Base):
    """告警单  detail 用 JSON 列存半结构化明细（关系库里塞文档型字段）"""
    __tablename__ = "alerts"
    id = Column(Integer, primary_key=True)
    device_id = Column(ForeignKey("devices.id"))
    ts = Column(DateTime, nullable=False)    # 播种时显式给值 → 结果完全可复现
    level = Column(String(8))               # warn / alarm
    detail = Column(JSON)
    device = relationship("Device", back_populates="alerts")


WAL_MODE = ""


def build_seed():
    """建库→建表→开WAL→灌种子（drop_all 先行，所以整个 notebook 可反复运行）"""
    eng = create_engine(SQLITE_URL,
                        connect_args={"timeout": 15, "check_same_thread": False})

    @event.listens_for(eng, "connect")
    def _open_wal(dbapi_conn, _record):      # 每条连接都切 WAL + 允许服务多线程读
        cur = dbapi_conn.cursor()
        cur.execute("PRAGMA journal_mode=WAL")
        cur.execute("PRAGMA busy_timeout=15000")
        cur.close()

    Base.metadata.drop_all(eng)
    Base.metadata.create_all(eng)
    with Session(eng) as s:
        pump3 = Device(code="PUMP-03", name="3号泵站循环泵", line_code="LINE-A2")
        fan2 = Device(code="FAN-02", name="2号主风机", line_code="LINE-B1")
        motor = Device(code="MOTOR-A2-1", name="A2线主电机", line_code="LINE-A2")
        s.add_all([pump3, fan2, motor])
        UNIT = {"vib_rms": "mm/s", "pressure": "MPa", "temp": "℃", "current": "A"}
        for d, tags in ((pump3, ["vib_rms", "pressure", "temp"]),
                        (fan2, ["vib_rms", "temp"]),
                        (motor, ["current", "vib_rms"])):
            for t in tags:
                d.points.append(SensorPoint(tag=t, unit=UNIT[t]))
        pump3.alerts.append(Alert(ts=datetime(2026, 9, 9, 9, 40), level="warn",
                                  detail={"vib_rms": 4.96, "msg": "振动连续3日爬升"}))
        pump3.alerts.append(Alert(ts=datetime(2026, 9, 9, 9, 55), level="alarm",
                                  detail={"vib_rms": 5.10, "msg": "超过基线 20%"}))
        fan2.alerts.append(Alert(ts=datetime(2026, 9, 8, 14, 5), level="warn",
                                 detail={"temp": 41.2, "msg": "温升偏快"}))
        s.commit()
    global WAL_MODE
    with eng.connect() as c:                 # 决策矩阵留痕：真的切成 WAL 了吗？
        WAL_MODE = str(c.execute(text("PRAGMA journal_mode")).scalar()).lower()
    print(f"[关系库] SQLite {sqlite_version()} | journal_mode={WAL_MODE} | 台账3台已播种")
    assert WAL_MODE == "wal", "WAL 未生效，决策矩阵结论不成立"
    return eng


def sqlite_version():
    import sqlite3
    return sqlite3.sqlite_version


ENGINE = build_seed()


[关系库] SQLite 3.50.2 | journal_mode=wal | 台账3台已播种


### Starter 3：`def device_profile(code)` —— **TODO-P7-1 关系台账查询【C层】**
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

设备 360° 视图 = `GET /api/device/{code}` 的数据源。要求写在 docstring 里，坑写在注释里：
**查不到的设备也要返回「有形状的空值」**（`{"found": False, ...}`），否则 KV 层没法缓存空结果 → 缓存穿透。
完成态只有 8 行：`select` 主表 → 顺 relationship 取从表 → 告警按时间倒序 → 组装 dict。

In [11]:
# -*- coding: utf-8 -*-
def device_profile(code: str) -> dict:
    """【P7-1 完成态】设备360°视图：台账 + 测点清单 + 近期告警

    服务层 `GET /api/device/{code}` 的数据源就在这里。
    ORM 写法要点：先 select 主表拿对象，再顺着 relationship 属性取从表，
    这就是"P6 用关系属性，不用手写 JOIN"的便利之处（代价是 N+1，见下方提示）。
    """
    with Session(ENGINE) as s:
        d = s.execute(select(Device).where(Device.code == code)).scalar_one_or_none()
        if d is None:
            return {"device": None, "found": False}
        return {
            "found": True,
            "device": {"code": d.code, "name": d.name, "line": d.line_code,
                       "installed": str(d.installed)[:10]},
            "points": [{"tag": p.tag, "unit": p.unit} for p in d.points],
            "point_count": len(d.points),
            "alerts": [{"ts": str(a.ts), "level": a.level, "detail": a.detail}
                       for a in sorted(d.alerts, key=lambda a: a.ts, reverse=True)],
            "alert_count": len(d.alerts),
        }


print(json.dumps(device_profile("PUMP-03"), ensure_ascii=False, indent=1))
print("查不存在的设备 →", device_profile("ROBOT-99")["found"], "（空值也要有形状，防缓存穿透）")


{
 "found": true,
 "device": {
  "code": "PUMP-03",
  "name": "3号泵站循环泵",
  "line": "LINE-A2",
  "installed": "2024-03-01"
 },
 "points": [
  {
   "tag": "vib_rms",
   "unit": "mm/s"
  },
  {
   "tag": "pressure",
   "unit": "MPa"
  },
  {
   "tag": "temp",
   "unit": "℃"
  }
 ],
 "point_count": 3,
 "alerts": [
  {
   "ts": "2026-09-09 09:55:00",
   "level": "alarm",
   "detail": {
    "vib_rms": 5.1,
    "msg": "超过基线 20%"
   }
  },
  {
   "ts": "2026-09-09 09:40:00",
   "level": "warn",
   "detail": {
    "vib_rms": 4.96,
    "msg": "振动连续3日爬升"
   }
  }
 ],
 "alert_count": 2
}
查不存在的设备 → False （空值也要有形状，防缓存穿透）


### Starter 4：存储引擎② 时序播种（不规则采样 + 明细落库 + 索引）
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

真实振动采集**从不按整点来**：采样间隔用 `rng.uniform(0.5, 1.5)` 秒抖动，7200 点、2 小时窗口。
三台设备的物理设定就是本课的"剧情数据"：
- `PUMP-03`：基线 4.0 + 后半段线性爬升 → **该判磨损**
- `FAN-02`：基线 3.0 纯噪声无趋势 → **该判正常**
- `MOTOR-A2-1`：轻微爬升 → 泵的下游客体
固定 `SEED=7` + `RandomState` + 显式时间戳，所以全班、每次运行看到的数字**一模一样**，换机也一样。

In [13]:
# -*- coding: utf-8 -*-
"""存储引擎② 时序：分钟级窗口特征（DuckDB 的离线等价实现 = pandas 列式）

时序场景三板斧（工单P7 · 决策矩阵第2行"分钟级窗口特征 DuckDB vs SQLite逐行"）：
  1) 不规则采样 → 2) ASOF 对齐整点 → 3) 时间窗口聚合(avg/max/slope/last) 喂 P6 模型
原厂写法（真实项目，机房无 duckdb 时下面两格自动换 pandas，**口径完全一致**）：
  SELECT device, avg(vib_rms), max(vib_rms), count(*) FROM vib
  WHERE ts >= (SELECT max(ts) FROM vib WHERE device=$dev) - INTERVAL $mins MINUTE
  GROUP BY device
"""
DEVICES_TS = [("PUMP-03", 4.00, 0.0200),      # (设备, 基线 mm/s, 后半程爬升 mm/s每分钟)
              ("FAN-02", 3.00, 0.0002),
              ("MOTOR-A2-1", 3.20, 0.0003)]
T0 = datetime(2026, 9, 9, 8, 0, 0)
N_SAMPLES = 7200               # 每台约2小时（间隔0.5~1.5s 的不规则采样，贴近真实OPC-UA突发）
NOISE = 0.06


def _gen_series(base, slope, rng, n=N_SAMPLES):
    """按物理口径合成一条振动序列：后半程线性爬升 + 高斯噪声（全部向量化）"""
    dt = rng.uniform(0.5, 1.5, n)                    # 采样间隔(秒) 不规则
    tsec = np.cumsum(dt)
    half = tsec[n // 2]
    ramp = np.where(tsec >= half, slope * (tsec - half) / 60.0, 0.0)
    vib = base + ramp + rng.normal(0.0, NOISE, n)
    ts = [T0 + timedelta(seconds=float(x)) for x in tsec]
    temp = 40.0 + (vib - base) * 3.0 + rng.normal(0.0, 0.4, n)   # 振动升→温度升（弱相关）
    return pd.DataFrame({"ts": ts, "device": "", "vib_rms": vib, "temp": temp})


def seed_timeseries():
    """播种：① pandas 列存快照（窗口特征用）② SQLite 原始逐行表（对照用）"""
    rng = np.random.RandomState(SEED)
    frames = []
    for dev, base, slope in DEVICES_TS:
        frames.append(_gen_series(base, slope, rng).assign(device=dev))
    df = pd.concat(frames, ignore_index=True).sort_values(["device", "ts"]).reset_index(drop=True)
    df.to_pickle(TS_PKL)
    with sqlite3.connect(SQLITE_FILE) as c:          # 同一库文件里再开一张原始表
        c.execute("DROP TABLE IF EXISTS vib_raw")
        c.execute("CREATE TABLE vib_raw(ts TEXT, device TEXT, vib_rms REAL, temp REAL)")
        c.executemany("INSERT INTO vib_raw VALUES(?,?,?,?)",
                      [(r.ts.isoformat(sep=" "), r.device, r.vib_rms, r.temp)
                       for r in df.itertuples(index=False)])
        c.execute("CREATE INDEX ix_vib_dev_ts ON vib_raw(device, ts)")   # 公平对比：给足索引
        c.commit()
    print(f"[时序库] 列存快照 {len(df)} 行 → {os.path.basename(TS_PKL)}"
          f" | SQLite 原始表 {len(df)} 行(已建 (device,ts) 索引)")
    return df


_TS_MEMO = {}     # 进程内复用，免得每次窗口计算都重读磁盘（= "什么值得缓存"的第一问）


def load_series(device: str) -> pd.DataFrame:
    if device not in _TS_MEMO:
        _TS_MEMO[device] = pd.read_pickle(TS_PKL).query("device == @device").reset_index(drop=True)
    return _TS_MEMO[device]


seed_timeseries()


[时序库] 列存快照 21600 行 → vib_columnar.pkl | SQLite 原始表 21600 行(已建 (device,ts) 索引)


### Starter 5：`def window_feats(device, minutes)` —— **TODO-P7-2 时序窗口特征【C层】**
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

模型吃的五个特征 `samples/avg_vib/max_vib/slope/last_vib` 全在这一格产出；
工单决策矩阵第 2 行的实测也发生在这里：同一份数据，**列式（pandas 向量化）** vs **逐行（SQLite 游标 + Python for）**，
`bench_window()` 真跑 5 次取最小值打印倍数。
两个坑：① `slope` 的分母要用**窗口真实跨度**，写死 `minutes` 在数据不足时会算错；
② 别只看 `FAN-02` 也有 slope——那是噪声骗出来的假斜率（本格末尾点名提醒，训练好的模型也知道）。

In [15]:
# -*- coding: utf-8 -*-
def window_feats(device: str, minutes: int = 10) -> dict:
    """【P7-2 完成态】最近 N 分钟窗口特征（模型推理的实时特征源）

    口径与 starter 的 DuckDB 版一一对应：
      n / avg_vib / max_vib / slope / last_vib
    slope = (窗口末值 - 窗口首值) / 窗口跨度(分钟) —— 只认"末值减首值"，
    这样噪声不会把趋势带偏（P2.2 学过的滚动斜率这里做了简化）。
    """
    df = load_series(device)
    if df.empty:
        return {"samples": 0, "avg_vib": 0.0, "max_vib": 0.0, "slope": 0.0, "last_vib": 0.0}
    cut = df["ts"].max() - timedelta(minutes=minutes)
    w = df[df["ts"] >= cut]                                  # 一次布尔掩码 = 列式过滤
    first_v, last_v = float(w["vib_rms"].iloc[0]), float(w["vib_rms"].iloc[-1])
    span_min = (w["ts"].iloc[-1] - w["ts"].iloc[0]).total_seconds() / 60.0 or 1.0
    return {"samples": int(len(w)),
            "avg_vib": round(w["vib_rms"].mean(), 3),
            "max_vib": round(w["vib_rms"].max(), 3),
            "slope": round((last_v - first_v) / span_min, 4),   # mm/s 每分钟
            "last_vib": round(last_v, 3)}


def window_feats_row_by_row(device: str, minutes: int = 10) -> dict:
    """反面教材（决策矩阵对照项）：同一份数据用 SQLite 逐行 + Python 循环算"""
    with sqlite3.connect(SQLITE_FILE) as c:
        c.row_factory = sqlite3.Row
        last_ts = c.execute("SELECT max(ts) m FROM vib_raw WHERE device=?", (device,)).fetchone()["m"]
        cut = (datetime.fromisoformat(last_ts) - timedelta(minutes=minutes)).isoformat(sep=" ")
        rows = c.execute("SELECT vib_rms FROM vib_raw WHERE device=? AND ts>=? ORDER BY ts",
                         (device, cut)).fetchall()
    vals = [r["vib_rms"] for r in rows]                      # 逐行反序列化成 Python 对象
    total = 0.0
    mx = -1e9
    for v in vals:                                           # 手工聚合 = 没有向量化
        total += v
        if v > mx:
            mx = v
    return {"samples": len(vals), "avg_vib": round(total / len(vals), 3), "max_vib": round(mx, 3)}


def bench_window(device="PUMP-03", minutes=120, reps=5):
    """列式 vs 逐行：全量2小时窗口的真实耗时对比（数字来自本机实测，非估算）"""
    def _time(fn, reps):
        best = float("inf")
        for _ in range(reps):
            t0 = time.perf_counter()
            fn(device, minutes)
            best = min(best, time.perf_counter() - t0)
        return best

    t_col = _time(window_feats, reps)
    t_row = _time(window_feats_row_by_row, reps)
    speedup = t_row / t_col if t_col > 0 else float("inf")
    print(f"[窗口基准] {device} 最近{minutes}分钟 / {window_feats(device, minutes)['samples']} 个采样点")
    print(f"  列式(pandas)  {t_col * 1000:7.3f} ms")
    print(f"  逐行(SQLite)  {t_row * 1000:7.3f} ms   → 相差 {speedup:4.1f} 倍")
    assert t_row > t_col, "本例应体现列式聚合优势，若反转请检查数据量"
    return {"columnar_ms": round(t_col * 1000, 3), "rowwise_ms": round(t_row * 1000, 3),
            "speedup": round(speedup, 1)}


print(json.dumps(window_feats("PUMP-03"), ensure_ascii=False))
print(json.dumps(window_feats("FAN-02"), ensure_ascii=False))
print("对比：PUMP-03 avg=5.1 且持续爬升；FAN-02 avg=3.0，它的 slope 只是噪声抖出来的假斜率")
print("     → 单看斜率会误判（A层特征重要度会量化这一点），模型靠 avg/峰值/末值 组合才稳")
BENCH_WIN = bench_window()


{"samples": 600, "avg_vib": 5.095, "max_vib": 5.36, "slope": 0.0217, "last_vib": 5.207}
{"samples": 600, "avg_vib": 3.011, "max_vib": 3.212, "slope": 0.0127, "last_vib": 3.015}
对比：PUMP-03 avg=5.1 且持续爬升；FAN-02 avg=3.0，它的 slope 只是噪声抖出来的假斜率
     → 单看斜率会误判（A层特征重要度会量化这一点），模型靠 avg/峰值/末值 组合才稳
[窗口基准] PUMP-03 最近120分钟 / 7200 个采样点
  列式(pandas)    0.537 ms
  逐行(SQLite)    8.791 ms   → 相差 16.4 倍


### Starter 6：ASOF 对齐与降采样（DuckDB `ASOF LEFT JOIN` 的 pandas 等价）
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

时序第二板斧：不规则采样要**对齐到整点**才画得出趋势线。原厂一句
`FROM grid ASOF LEFT JOIN vib v ON v.device=? AND g >= v.ts`，pandas 等价就是
`pd.merge_asof(..., direction="backward")`——本格拿 5 个整点把「取整点前最近一条」演给你看，
再用 `resample("10min")` 降采样。**这是本课唯一一处「两种引擎逐字对照」的格，读懂它就不怕换引擎。**

In [17]:
# -*- coding: utf-8 -*-
def asof_grid(device: str, show: int = 5) -> list:
    """ASOF 对齐演示：把不规则采样对齐到"每分钟整点"，每点取整点前最近一条

    DuckDB 写法:  FROM grid ASOF LEFT JOIN vib v ON v.device=? AND g >= v.ts
    pandas 等价:  pd.merge_asof(grid, samples, on='g', direction='backward')
    —— 换引擎不换语义，这就是"离线等价实现"的判据。
    """
    df = load_series(device)
    grid = pd.DataFrame({"g": pd.date_range(df["ts"].min().floor("min"),
                                            df["ts"].max().floor("min"), freq="min")})
    samples = df.assign(g=df["ts"])[["g", "ts", "vib_rms"]].sort_values("g").reset_index(drop=True)
    aligned = pd.merge_asof(grid, samples, on="g", direction="backward")
    return [{"grid": str(r["g"]), "ts": "—" if pd.isna(r["ts"]) else str(r["ts"]),
             "vib": None if pd.isna(r["vib_rms"]) else round(float(r["vib_rms"]), 3)}
            for _, r in aligned.head(show).iterrows()]


def downsample(df: pd.DataFrame, minutes: int = 10) -> pd.DataFrame:
    """time_bucket 降采样：不规则原始序列 → 每 N 分钟均值/峰值（DuckDB 的 date_trunc 等价）"""
    g = df.set_index("ts").resample(f"{minutes}min")
    return pd.DataFrame({"avg": g["vib_rms"].mean().round(3),
                         "max": g["vib_rms"].max().round(3),
                         "n": g["vib_rms"].size()}).dropna()


print("ASOF 对齐前 5 个整点（原始采样间隔 0.5~1.5s 不等，整点上通常没有恰好同刻的采样）：")
print(json.dumps(asof_grid("PUMP-03"), ensure_ascii=False, indent=1))
print("\n降采样（time_bucket 10 分钟）前 3 桶：")
print(downsample(load_series("PUMP-03"), 10).head(3).to_string())


ASOF 对齐前 5 个整点（原始采样间隔 0.5~1.5s 不等，整点上通常没有恰好同刻的采样）：
[
 {
  "grid": "2026-09-09 08:00:00",
  "ts": "—",
  "vib": null
 },
 {
  "grid": "2026-09-09 08:01:00",
  "ts": "2026-09-09 08:00:59.785727",
  "vib": 4.053
 },
 {
  "grid": "2026-09-09 08:02:00",
  "ts": "2026-09-09 08:01:59.373904",
  "vib": 4.047
 },
 {
  "grid": "2026-09-09 08:03:00",
  "ts": "2026-09-09 08:02:59.796421",
  "vib": 4.061
 },
 {
  "grid": "2026-09-09 08:04:00",
  "ts": "2026-09-09 08:03:59.393665",
  "vib": 3.914
 }
]

降采样（time_bucket 10 分钟）前 3 桶：
                       avg    max    n
ts                                    
2026-09-09 08:00:00  3.999  4.185  601
2026-09-09 08:10:00  4.002  4.203  605
2026-09-09 08:20:00  3.997  4.203  606


### Starter 7：存储引擎③ 拓扑图（邻接表 + 变长跳遍历）
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

拓扑关系：厂区 → 产线 → 设备，外加一条 `上游供电` 边（泵故障会连累同一供电回路的电机与数控中心）。
图库真正值钱的是 **变长多跳**：Cypher `-[:上游供电*1..3]->` 一句直达；关系库要写三层自连接再去重。
本格的 `follow(start, rel, hops=(1,3))` 就是那句 Cypher 的 BFS 等价实现，节点属性与边关系都落 json 持久化。

In [19]:
# -*- coding: utf-8 -*-
"""存储引擎③ 图：拓扑影响路径（Kùzu 嵌入式的离线等价 = 邻接表 + BFS）

为什么台账放关系库、拓扑放图库？——"停 3 号泵要联动停哪些线/哪些设备"这种
**变长多跳**问题，关系库要写 3 层自连接再加去重，图库一句 Cypher 直达：

  MATCH (d:设备 {id:$dev})-[:上游供电*1..3]->(x:设备)
  OPTIONAL MATCH (l:产线)-[:有设备]->(x)
  RETURN x.id, x.name, collect(l.id)

本 notebook 的 Kùzu 不可用，于是把同一张图存成 nodes/edges 两份 JSON，
用 `follow()` 做可变跳遍历：`hops=(1,3)` 就是 Cypher 里的 `*1..3`。
"""
NODES = [("厂区", "P1", "两江智造一号厂区", None),
         ("产线", "LINE-A2", "精加工A2线", None),
         ("产线", "LINE-B1", "装配B1线", None),
         ("产线", "LINE-C3", "冲压C3线", None),
         ("设备", "PUMP-03", "3号泵站循环泵", "warn"),
         ("设备", "MOTOR-A2-1", "A2线主电机", "ok"),
         ("设备", "CNC-A2-1", "A2线数控中心", "ok"),
         ("设备", "FAN-02", "2号主风机", "ok"),
         ("设备", "PUMP-01", "1号泵站循环泵", "ok"),
         ("测点", "PUMP-03.vib_rms", "振动测点", None),
         ("测点", "PUMP-03.temp", "温度测点", None)]
EDGES = [("P1", "有产线", "LINE-A2"), ("P1", "有产线", "LINE-B1"), ("P1", "有产线", "LINE-C3"),
         ("LINE-A2", "有设备", "PUMP-03"), ("LINE-A2", "有设备", "MOTOR-A2-1"),
         ("LINE-A2", "有设备", "CNC-A2-1"), ("LINE-B1", "有设备", "FAN-02"),
         ("LINE-C3", "有设备", "PUMP-01"),
         ("PUMP-03", "上游供电", "MOTOR-A2-1"), ("MOTOR-A2-1", "上游供电", "CNC-A2-1"),
         ("PUMP-03", "有测点", "PUMP-03.vib_rms"), ("PUMP-03", "有测点", "PUMP-03.temp")]


def seed_graph():
    """建图落盘：邻接表(out/in) + 节点属性表，等价 Kùzu 的 (节点表, 关系表)"""
    out, inc, props = {}, {}, {}
    for label, nid, name, state in NODES:
        props[nid] = {"label": label, "id": nid, "name": name, "state": state}
    for src, rel, dst in EDGES:
        out.setdefault((src, rel), []).append(dst)
        inc.setdefault((dst, rel), []).append(src)
    with open(GRAPH_JSON, "w", encoding="utf-8") as f:
        json.dump({"nodes": props, "out": {f"{a}|{b}": v for (a, b), v in out.items()},
                   "in": {f"{a}|{b}": v for (a, b), v in inc.items()}}, f, ensure_ascii=False)
    print(f"[图库] 节点{len(NODES)} 关系{len(EDGES)} → {os.path.basename(GRAPH_JSON)}"
          f"（Kùzu 离线替代：邻接表 + 可变跳遍历）")
    return props, out, inc


GRAPH_PROPS, GRAPH_OUT, GRAPH_IN = seed_graph()


def follow(start: str, rel: str, hops=(1, 3)) -> list:
    """可变跳遍历 = Cypher 的 -[:rel*hops[0]..hops[1]]-> ；BFS 去重防环"""
    seen, frontier, found, depth = set(), [start], [], 0
    for depth in range(1, hops[1] + 1):
        nxt = []
        for node in frontier:
            for nb in GRAPH_OUT.get((node, rel), []):
                if nb not in seen:
                    seen.add(nb)
                    found.append({"id": nb, "hop": depth})
                    nxt.append(nb)
        frontier = nxt
    return [f for f in found if f["hop"] >= hops[0]]


def lines_of(dev_id: str) -> list:
    """反查：某设备挂在哪条产线上 = Cypher 的 (l:产线)-[:有设备]->(d)"""
    return GRAPH_IN.get((dev_id, "有设备"), [])


[图库] 节点11 关系12 → topology.json（Kùzu 离线替代：邻接表 + 可变跳遍历）


### Starter 8：`def impact_path(device_id, max_hops)` —— **TODO-P7-3 影响路径【B层】**
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

运维决策的核心查询：「停 3 号泵要联动停哪些线、哪些设备」。docstring 里保留了原厂 Kùzu Cypher 全文，
你只需要用给好的 `follow()` / `lines_of()` 把它翻译出来。
**坑（也是 kuzu 高频翻车点）**：不要把 `有设备` 当出边跟着走（那是产线→设备方向），要用 `lines_of()` 反查。
验收用了个好办法：`closure_reachable()` 暴力闭包算真值，再断言你的 BFS **覆盖率 = 100%**——图算法也能有单元测试。

In [21]:
# -*- coding: utf-8 -*-
def impact_path(device_id: str, max_hops: int = 3) -> dict:
    """【P7-3 完成态】影响路径：这台设备异常会波及哪些下游设备 / 哪些产线

    原厂 Cypher（Kùzu 支持，本机离线用 follow() 等价遍历）：
      MATCH (d:设备 {id:$dev})-[:上游供电*1..3]->(x:设备)
      OPTIONAL MATCH (l:产线)-[:有设备]->(x)
      RETURN x.id, x.name, collect(l.id)
    """
    downstream = follow(device_id, "上游供电", (1, max_hops))
    collateral = [{"downstream": n["id"], "down_name": GRAPH_PROPS[n["id"]]["name"],
                   "hop": n["hop"], "lines": lines_of(n["id"])} for n in downstream]
    hit_lines = sorted({l for c in collateral for l in c["lines"]})
    return {"device": device_id,
            "own_lines": lines_of(device_id),
            "collateral": collateral,
            "affected_lines": sorted(set(hit_lines) | set(lines_of(device_id))),
            "downstream_count": len(collateral)}


def closure_reachable(start: str, rel: str) -> set:
    """暴力闭包（对照实现）：反复扩展直到不再新增——用来验证 BFS 的覆盖率"""
    reach, frontier = set(), {start}
    while frontier:
        nxt = {nb for n in frontier for nb in GRAPH_OUT.get((n, rel), [])} - reach - {start}
        reach |= nxt
        frontier = nxt
    return reach


rep = impact_path("PUMP-03")
print(json.dumps(rep, ensure_ascii=False, indent=1))
truth = closure_reachable("PUMP-03", "上游供电")
got = {c["downstream"] for c in rep["collateral"]}
coverage = len(got & truth) / len(truth) * 100 if truth else 100.0
print(f"[B层覆盖闸] 下游真值{sorted(truth)} | BFS命中{sorted(got)} | 影响面覆盖率 {coverage:.1f}%")
assert got == truth, "影响路径漏检，图谱遍历不完整"
assert coverage == 100.0, f"覆盖率未达 100%（当前{coverage}%）"
assert "MOTOR-A2-1" in got and "LINE-A2" in rep["affected_lines"], "泵→电机→A2线联动链断裂"
print("空载对照 FAN-02 →", json.dumps(impact_path("FAN-02"), ensure_ascii=False))


{
 "device": "PUMP-03",
 "own_lines": [
  "LINE-A2"
 ],
 "collateral": [
  {
   "downstream": "MOTOR-A2-1",
   "down_name": "A2线主电机",
   "hop": 1,
   "lines": [
    "LINE-A2"
   ]
  },
  {
   "downstream": "CNC-A2-1",
   "down_name": "A2线数控中心",
   "hop": 2,
   "lines": [
    "LINE-A2"
   ]
  }
 ],
 "affected_lines": [
  "LINE-A2"
 ],
 "downstream_count": 2
}
[B层覆盖闸] 下游真值['CNC-A2-1', 'MOTOR-A2-1'] | BFS命中['CNC-A2-1', 'MOTOR-A2-1'] | 影响面覆盖率 100.0%
空载对照 FAN-02 → {"device": "FAN-02", "own_lines": ["LINE-B1"], "collateral": [], "affected_lines": ["LINE-B1"], "downstream_count": 0}


### Starter 9：存储引擎④ 案例向量库 + `def search(query, k, device)` —— **TODO-P7-4 语义检索【C层】**
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

四条历史故障案例 → 中文哈希嵌入（字×1 + bigram×2，L2 归一化）→ `.npz` 持久化 → 余弦近邻。
**这就是 Chroma 那套流程，只是不下载模型**（工单速查表里"chroma 首次卡住=默认嵌入要联网"，本课用哈希嵌入绕开）。
两个必须掌握的口径：① 向量已归一化 → **点积即余弦**；② `dist = 1 - 相似度`（Chroma 口径，排反序就选错案例）。
本格的演示很狠：纯语义会把 Top1 给**最长的那条**案例（长度偏置），加上 `device` 元数据过滤才修正——
`where=` 不是可选糖，是召回率的命。

In [23]:
# -*- coding: utf-8 -*-
"""存储引擎④ 向量：故障案例语义检索（Chroma 的离线等价 = numpy 余弦 + npz 持久化）

离线要点：不下载任何嵌入模型。自实现"中文哈希嵌入"——字(权1) + bigram(权2) → 定长向量，
   中文双字成词，bigram 加权能明显压住单字的噪声匹配。
语义检索流程: 案例文本 → 哈希嵌入 → npz 持久化 → 余弦近邻召回（= Chroma 的那套流程）
A层升级位（工单P7-A层）: 换 bge-small-zh / text2vec 本地模型，**接口不变**；
   本课先实测"元数据过滤"这条零成本增益（见 A层拓展1 的三方案对照）。
⚠ 探究点：为什么用 zlib.crc32 而不是内置 hash()？
   CPython 对 str 的 hash() 带 PYTHONHASHSEED 随机化，同一句话每次进程得到的桶都不同，
   索引文件就废了（缓存/向量库都要求"可复现"）。这是工单"常见错误速查"之外的第6个坑。
"""
import zlib

DIM = 256
CASES = [
    {"id": "case-001", "text": "3号泵站振动爬升 轴承磨损 特征频率吻合 更换轴承后恢复",
     "meta": {"device": "PUMP-03", "action": "更换轴承", "year": 2025}},
    {"id": "case-002", "text": "循环泵压力波动 叶轮堵塞 清洗叶轮 振动恢复正常",
     "meta": {"device": "PUMP-03", "action": "清洗叶轮", "year": 2024}},
    {"id": "case-003", "text": "主风机振动超标 地脚螺栓松动 紧固后温度振动下降",
     "meta": {"device": "FAN-02", "action": "紧固螺栓", "year": 2025}},
    {"id": "case-004", "text": "电机电流异常 联轴器对中不良 激光对中后电流恢复",
     "meta": {"device": "MOTOR-A2-1", "action": "激光对中", "year": 2023}},
]
# A层评测集：每条查询配"标准答案案例 + 台账里已知的设备"（召回率拿它算，不靠感觉）
QREL = [("泵振动持续变大如何处理", "case-001", "PUMP-03"),
        ("风机振动超标怎么办", "case-003", "FAN-02"),
        ("叶轮堵了压力不稳", "case-002", "PUMP-03"),
        ("联轴器没对中电流异常", "case-004", "MOTOR-A2-1"),
        ("轴承磨损该换什么", "case-001", "PUMP-03"),
        ("振动大温度也高", "case-003", "FAN-02")]


def embed_hash(text: str, dim: int = DIM) -> np.ndarray:
    """确定性中文嵌入：字级 + bigram（双字成词，权重×2）哈希到固定维度后 L2 归一化"""
    v = np.zeros(dim)
    for ch in text:
        v[zlib.crc32(ch.encode("utf-8")) % dim] += 1.0
    for i in range(len(text) - 1):
        v[zlib.crc32(text[i:i + 2].encode("utf-8")) % dim] += 2.0
    return v / (np.linalg.norm(v) or 1.0)


def seed_vector():
    """建索引并持久化（= chroma.PersistentClient + col.upsert）"""
    mat = np.vstack([embed_hash(c["text"]) for c in CASES])
    np.savez(VEC_NPZ, ids=[c["id"] for c in CASES], texts=[c["text"] for c in CASES],
             metas=[json.dumps(c["meta"], ensure_ascii=False) for c in CASES], vecs=mat)
    print(f"[向量库] {len(CASES)} 条案例 / 维度{DIM} → {os.path.basename(VEC_NPZ)}"
          f"（Chroma 离线替代：npz + numpy 余弦）")


def search(query: str, k: int = 2, device: str = None) -> list:
    """【P7-4 完成态】语义近邻召回：'振动持续变大' 也能召回 '振动爬升' 案例

    device 参数 = Chroma 的 `where={"device": ...}` 元数据过滤：
    工厂案例库先按设备缩小货架、再做语义排序，既能防"别的泵的处置方案串味"，
    又比"全库语义硬排"更准（智能体取证时就是把台账查到的设备号带进来）。
    """
    idx = np.load(VEC_NPZ, allow_pickle=True)
    metas = [json.loads(str(m)) for m in idx["metas"]]
    cand = [i for i, m in enumerate(metas) if device is None or m.get("device") == device]
    cand = cand or list(range(len(metas)))                   # 过滤空了要能回退全库
    q = embed_hash(query)
    sims = idx["vecs"][cand] @ q                             # 已归一化 → 点积即余弦
    order = [cand[i] for i in np.argsort(-sims)[:k]]
    return [{"case": str(idx["texts"][i]), "id": str(idx["ids"][i]), "meta": metas[i],
             "dist": round(float(1 - sims[cand.index(i)]), 4)}   # 距离=1-相似度（Chroma口径）
            for i in order]


seed_vector()
print("纯语义检索 Top2：", json.dumps(search("泵振动持续变大怎么办", 2), ensure_ascii=False))
print("带设备元数据过滤 Top2：", json.dumps(
    [c["id"] for c in search("泵振动持续变大怎么办", 2, device="PUMP-03")], ensure_ascii=False))
print("→ 探究点：哈希词袋按 gram 数归一化，长文档会被短文档挤掉（长度偏置）。"
      "工厂场景先按设备缩货架再排序，既符合直觉又零成本纠偏——A层会量化这条增益。")


[向量库] 4 条案例 / 维度256 → fault_cases.npz（Chroma 离线替代：npz + numpy 余弦）
纯语义检索 Top2： [{"case": "主风机振动超标 地脚螺栓松动 紧固后温度振动下降", "id": "case-003", "meta": {"device": "FAN-02", "action": "紧固螺栓", "year": 2025}, "dist": 0.7851}, {"case": "3号泵站振动爬升 轴承磨损 特征频率吻合 更换轴承后恢复", "id": "case-001", "meta": {"device": "PUMP-03", "action": "更换轴承", "year": 2025}, "dist": 0.8798}]
带设备元数据过滤 Top2： ["case-001", "case-002"]
→ 探究点：哈希词袋按 gram 数归一化，长文档会被短文档挤掉（长度偏置）。工厂场景先按设备缩货架再排序，既符合直觉又零成本纠偏——A层会量化这条增益。


### Starter 10：存储引擎⑤ KV 缓存（Redis 优先 / 离线 TTL 兜底）
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

同一接口两种实现，是工程决策课的活教材：`RedisKV`（真连，跨进程共享 + 原生 TTL + `INCR` 原子计数）
和 `TTLKV`（纯标准库 json 落盘 + 进程内 dict + 锁）。`make_kv()` 默认走离线，
**即使本机装了 redis 也打印一行"不启用：课堂结果必须可复现"**——可复现性优先于炫技。
`_NOW` 假时钟是这格的机关：注入时间后 TTL 过期当场可见，不用 `sleep(30)` 干等。

In [25]:
# -*- coding: utf-8 -*-
"""存储引擎⑤ KV缓存：Redis 优先 / 离线等价的 TTL 磁盘 KV 兜底

同一接口两种实现（工程决策课）：
  Redis   —— 跨进程共享、原生 TTL、INCR 原子计数（限流）
  本兜底实现 —— 纯标准库 json 落盘 + 进程内 dict + 锁，机房零服务依赖
⚠ 诚实边界：兜底版只在**本进程内**共享，落盘文件仅作留痕/重启预热；
   多进程真共享必须上 Redis（真实项目部署时把 make_kv() 开关打开即可，接口不变）。

缓存三问（B04）：①什么值得缓存 ②锁多久(TTL) ③过期谁负责(失效策略)
"""
import threading

_NOW = [0.0]                                  # 可注入时钟：让 TTL 过期演示不用真睡觉


class TTLKV:
    name = "ttl-kv(离线兜底)"

    def __init__(self, path, clock=None):
        self.path = path
        self.clock = clock or (lambda: _NOW[0])
        self._d = {}                          # key -> (value, expire_at)
        self._lock = threading.Lock()
        self.hits = self.misses = 0

    def put(self, k, v, ttl=60):
        with self._lock:
            self._d[k] = (v, self.clock() + ttl)
            self._flush()
        return True

    def get(self, k):
        with self._lock:
            hit = self._d.get(k)
            if hit is None:
                self.misses += 1
                return None
            v, exp = hit
            if exp <= self.clock():           # 惰性失效：读到时已过期就当没有
                self._d.pop(k, None)
                self.misses += 1
                return None
            self.hits += 1
            return v

    def delete(self, k):
        with self._lock:
            self._d.pop(k, None)
            self._flush()

    def ttl_left(self, k):
        v = self._d.get(k)
        return None if v is None else round(v[1] - self.clock(), 2)

    def incr(self, k, ttl=3600):
        """原子计数：API 限流（每调用方每小时 N 次）"""
        with self._lock:
            n = int(self.get_raw(k) or 0) + 1
            self._d[k] = (n, self.clock() + ttl)
            self._flush()
            return n

    def get_raw(self, k):
        v = self._d.get(k)
        return None if v is None or v[1] <= self.clock() else v[0]

    def _flush(self):                         # 留痕：缓存内容+过期时刻可读（答辩可查）
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump({k: {"v": str(v), "exp": round(e, 2)} for k, (v, e) in self._d.items()},
                      f, ensure_ascii=False, indent=1)


class RedisKV(TTLKV):                          # 真实部署用（本机课堂不启用，见 make_kv）
    name = "redis"

    def __init__(self, url, ttl_clock=None):
        import redis                          # 延迟导入：没装也不影响离线运行
        self.r = redis.Redis.from_url(url, socket_connect_timeout=1, decode_responses=True)
        self.r.ping()
        self.path, self.clock, self._d = "/dev/null", (lambda: 0.0), {}
        self.hits = self.misses = 0
        self._lock = threading.Lock()

    def put(self, k, v, ttl=60):
        return self.r.setex(k, ttl, v)

    def get(self, k):
        return self.r.get(k)

    def delete(self, k):
        self.r.delete(k)

    def incr(self, k, ttl=3600):
        n = self.r.incr(k)
        if n == 1:
            self.r.expire(k, ttl)
        return n


def make_kv():
    """工厂 + 决策留痕：默认走离线兜底（可复现）；真实机房把 P77_USE_REDIS=1 切 Redis"""
    if os.environ.get("P77_USE_REDIS") == "1":
        try:
            kv = RedisKV(os.environ.get("REDIS_URL", "redis://127.0.0.1:6399/0"))
            print(f"[kv] 后端=redis（跨进程共享，真实部署选项）")
            return kv
        except Exception as e:
            print(f"[kv] redis 不可达({type(e).__name__}) → 回退离线 TTL-KV")
    print(f"[kv] 后端=ttl-kv 离线兜底 → {os.path.relpath(KV_FILE, BASE)}"
          f"（Redis 即使本机可导入也不启用：课堂结果必须可复现）")
    return TTLKV(KV_FILE)


KV = make_kv()


[kv] 后端=ttl-kv 离线兜底 → w09_scratch/data/kv_cache.json（Redis 即使本机可导入也不启用：课堂结果必须可复现）


### Starter 11：缓存三问实操（B04）——什么缓存 / 存多久 / 谁负责过期
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

三问一次跑完：① 重复且贵的（模型推理、台账聚合）才缓存，**实时告警绝不缓存**；
② TTL 用 `CACHE_TTL + jitter`——全班同时过期就是**缓存雪崩**，抖动是 3 行代码的事；
③ 过期由**读方惰性判定**（谁读到谁发现），外加"空值也缓存"防**穿透**。
最后用 `KV.incr()` 数一数限流计数器——这就是服务层那条 429 的地基。

In [27]:
# -*- coding: utf-8 -*-
# 缓存三问（B04）实操验证：注入假时钟，TTL 过期"当场可见"，不用 sleep 干等
_NOW[0] = 1000.0                              # 把逻辑时钟拨到 t=1000s

print("① 什么值得缓存？→ 重复且贵的计算（模型推理/台账聚合）；实时告警绝不缓存")
KV.put("demo:pred", "proba=0.91", ttl=30)
print("   put 后 get   :", KV.get("demo:pred"), "| 剩余TTL", KV.ttl_left("demo:pred"))
_NOW[0] += 29                                 # 时钟拨快 29s
print("   TTL-1s 再 get:", KV.get("demo:pred"), "| 剩余TTL", KV.ttl_left("demo:pred"))
print("\n② 锁多久？→ 大屏轮询类 10s、案例检索 5min、限流计数 1h；TTL 越长越省算力但越容易脏")
_NOW[0] += 2                                  # 越过 30s
print("   TTL+2s 再 get:", KV.get("demo:pred"), "（惰性失效：读到时已过期 → 回源）")

print("\n③ 过期谁负责？→ 读时惰性删除(本实现) / Redis 定时采样+过期回调")
print("   穿透兜底：查不到的设备也写一个短 TTL 空值，避免同一 key 反复打库")
KV.put("dev:ROBOT-99", json.dumps({"found": False}), ttl=5)
print("   空值命中 →", KV.get("dev:ROBOT-99"))

print("\n防雪崩：同一批 key 的 TTL 加随机抖动，别让它们在同一秒集体过期")
rng = np.random.RandomState(SEED)
ttls = [round(CACHE_TTL_DEV + rng.uniform(0, CACHE_JITTER), 2) for _ in range(6)]
print("   基线10s + 抖动 → 实际写入TTL:", ttls)

print("\n限流计数 incr（原子）:", [KV.incr("rl:demo-client") for _ in range(3)],
      "→ 超过阈值即拒绝服务(见服务层 429)")
print(f"命中率统计 hits={KV.hits} misses={KV.misses}")
assert KV.get("demo:pred") is None, "TTL 过期未生效"
print("✔ 缓存三问全部当场验证通过")


① 什么值得缓存？→ 重复且贵的计算（模型推理/台账聚合）；实时告警绝不缓存
   put 后 get   : proba=0.91 | 剩余TTL 30.0
   TTL-1s 再 get: proba=0.91 | 剩余TTL 1.0

② 锁多久？→ 大屏轮询类 10s、案例检索 5min、限流计数 1h；TTL 越长越省算力但越容易脏
   TTL+2s 再 get: None （惰性失效：读到时已过期 → 回源）

③ 过期谁负责？→ 读时惰性删除(本实现) / Redis 定时采样+过期回调
   穿透兜底：查不到的设备也写一个短 TTL 空值，避免同一 key 反复打库
   空值命中 → {"found": false}

防雪崩：同一批 key 的 TTL 加随机抖动，别让它们在同一秒集体过期
   基线10s + 抖动 → 实际写入TTL: [10.38, 13.9, 12.19, 13.62, 14.89, 12.69]

限流计数 incr（原子）: [1, 2, 3] → 超过阈值即拒绝服务(见服务层 429)
命中率统计 hits=3 misses=1
✔ 缓存三问全部当场验证通过


### Starter 12：诊断模型入库（复用 P6 五步流 + **F1(macro)≥0.80** 闸）
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

P7 不重造模型，**P7 复用 P6 的五步流**：划分→训练→交叉验证→调参→保存。
训练集用**同一套物理合成公式 + 同一个窗口统计函数**生成——特征口径漂移是这类项目最常见的事故
（训练用整点、推理用滑动窗口，线上必崩）。
`GridSearchCV(cv=5)` 打完分后本格还打印 **feature_importances**：你会看到斜率几乎没被用上，
因为**噪声也能造出假斜率**——这直接解释了 Starter 5 末尾的提醒，也是 P6 抽查「过拟合信号」的答案。
最后 `assert macro_f1 >= 0.80` 才允许 `joblib.dump`：**达标线写在代码里，不写在自评表上**。

In [29]:
# -*- coding: utf-8 -*-
"""诊断模型入库：复用 P6 五步流（划分→训练→交叉验证→调参→保存）

⚠ 特征口径必须与 tools.model_predict 完全一致：
   [avg_vib, max_vib, slope, last_vib, samples/100]
训练集用"同一套物理合成公式 + 同一个窗口统计函数"生成 —— 口径漂移是这类项目最常见的事故。
"""
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split


def _win_stats(vib, span_min):
    """与 window_feats 同一套算法的最小实现（特征口径只有一份）"""
    return [vib.mean(), vib.max(), (vib[-1] - vib[0]) / span_min, vib[-1]]


def synth_window_dataset(n_dev=240, minutes=10.0, n=600):
    """合成 240 台"虚拟设备"的 10 分钟窗口 → 特征矩阵 + 标签(0正常/1磨损)"""
    rng = np.random.RandomState(SEED)
    X, y = [], []
    for _ in range(n_dev):
        wear = rng.rand() < 0.5
        start = rng.uniform(4.45, 5.8) if wear else rng.uniform(2.8, 4.6)   # 窗口起点 mm/s
        slope = rng.uniform(0.005, 0.020) if wear else rng.uniform(-0.010, 0.018)  # 健康类的"斜率"多半是噪声
        noise = rng.uniform(0.03, 0.09)            # 窗口内噪声水平也随机
        t = np.linspace(0, minutes, n)
        vib = start + slope * t + rng.normal(0, noise, n)
        X.append(_win_stats(vib, minutes) + [n / 100.0])
        y.append(int(wear))
    return np.array(X), np.array(y)


def train_and_save():
    X, y = synth_window_dataset()
    Xtr, Xte, ytr, yte = train_test_split(X, y, stratify=y, random_state=SEED)
    grid = GridSearchCV(RandomForestClassifier(random_state=SEED),
                        {"n_estimators": [50, 150], "max_depth": [6, 10]}, cv=5, scoring="f1")
    grid.fit(Xtr, ytr)
    clf = grid.best_estimator_
    pred = clf.predict(Xte)
    macro_f1 = f1_score(yte, pred, average="macro")
    print("五步流 · 最优参数:", grid.best_params_)
    print("特征重要度 [avg, max, slope, last, n/100]:",
          np.round(clf.feature_importances_, 3),
          "→ 斜率几乎没用(噪声骗人)，主力是均值/峰值/末值（呼应P2.2：斜率要平滑才可靠）")
    print("混淆矩阵\n", confusion_matrix(yte, pred))
    print(classification_report(yte, pred, target_names=["正常", "磨损"], digits=3))
    assert macro_f1 >= 0.80, f"P6 抽查线未达标 F1(macro)={macro_f1:.3f}"
    joblib.dump(clf, CLF_FILE, compress=3)
    print(f"模型落盘 {os.path.relpath(CLF_FILE, BASE)} | F1(macro)={macro_f1:.3f} ≥ 0.80 ✔")
    return clf, round(float(macro_f1), 3)


MODEL, MODEL_F1 = train_and_save()


五步流 · 最优参数: {'max_depth': 6, 'n_estimators': 50}
特征重要度 [avg, max, slope, last, n/100]: [0.402 0.289 0.02  0.289 0.   ] → 斜率几乎没用(噪声骗人)，主力是均值/峰值/末值（呼应P2.2：斜率要平滑才可靠）
混淆矩阵
 [[26  3]
 [ 0 31]]
              precision    recall  f1-score   support

          正常      1.000     0.897     0.945        29
          磨损      0.912     1.000     0.954        31

    accuracy                          0.950        60
   macro avg      0.956     0.948     0.950        60
weighted avg      0.954     0.950     0.950        60

模型落盘 w09_scratch/data/models/p77_clf.joblib | F1(macro)=0.950 ≥ 0.80 ✔


### Starter 13：智能体工具箱（5 库 + 1 模型 → OpenAI tools 协议）
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

把企业资产接进智能体的**标准接口形态**：每个工具 = 一层薄封装 + 一段给大模型看的 `description`。
`description` 写不清楚，模型就选错工具——这就是"提示工程即接口文档"。
`model_predict` 里能看到 `MISS→HIT`：工具自己会走缓存，模型只管调用。
`REGISTRY` 是白名单：调一个不存在的工具名**故意抛 KeyError**（幻觉/越权工具的兜底），
服务层那条 500 兜底就是为它准备的。

In [31]:
# -*- coding: utf-8 -*-
"""智能体工具箱：把 5 种存储 + 1 个模型包成大模型可调用的 function（OpenAI tools 协议）

每个工具 = 存储引擎的一层薄封装，`description` 是给大模型看的"说明书"——
写不清楚，模型就选错工具。这是"把企业资产接进智能体"的标准接口形态。
"""
_CLF = None


def _clf():
    global _CLF
    if _CLF is None:                       # 进程内加载一次（= 服务启动时 warm-up）
        _CLF = joblib.load(CLF_FILE)
    return _CLF


def model_predict(device: str) -> dict:
    """缓存优先推理：同设备 TTL 内重复问直接命中 KV —— 演示缓存的真实价值"""
    ck = f"pred:{device}"
    hit = KV.get(ck)
    if hit:
        return {"device": device, "pred": json.loads(hit), "cache": "HIT"}
    feats = window_feats(device)
    x = np.array([[feats["avg_vib"], feats["max_vib"], feats["slope"],
                   feats["last_vib"], feats["samples"] / 100.0]])
    proba = _clf().predict_proba(x)[0]
    out = {"label": int(proba[1] > 0.5), "proba_wear": round(float(proba[1]), 3), "feats": feats}
    KV.put(ck, json.dumps(out, ensure_ascii=False), ttl=CACHE_TTL_PRED)
    return {"device": device, "pred": out, "cache": "MISS"}


TOOLS_SPEC = [
    {"type": "function", "function": {
        "name": "query_relational", "description": "查设备台账：名称/产线/测点/历史告警",
        "parameters": {"type": "object", "properties": {"device": {"type": "string"}},
                       "required": ["device"]}}},
    {"type": "function", "function": {
        "name": "query_timeseries", "description": "查设备最近N分钟时序特征：均值/峰值/斜率/最新值",
        "parameters": {"type": "object",
                       "properties": {"device": {"type": "string"}, "minutes": {"type": "integer"}},
                       "required": ["device"]}}},
    {"type": "function", "function": {
        "name": "impact_path", "description": "图数据库影响路径：设备异常波及哪些下游设备与产线",
        "parameters": {"type": "object", "properties": {"device": {"type": "string"}},
                       "required": ["device"]}}},
    {"type": "function", "function": {
        "name": "semantic_search", "description": "向量库语义检索历史故障案例与处置动作(可按设备过滤)",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string"}, "device": {"type": "string"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "model_predict", "description": "诊断模型推理：时序特征→磨损概率（结果写缓存）",
        "parameters": {"type": "object", "properties": {"device": {"type": "string"}},
                       "required": ["device"]}}},
]

REGISTRY = {
    "query_relational": lambda a: device_profile(a["device"]),
    "query_timeseries": lambda a: window_feats(a["device"], a.get("minutes", 10)),
    "impact_path": lambda a: impact_path(a["device"]),
    "semantic_search": lambda a: search(a["query"], 2, a.get("device")),
    "model_predict": lambda a: model_predict(a["device"]),
}


def run_tool(name: str, arguments: dict) -> dict:
    if name not in REGISTRY:
        raise KeyError(f"未注册的工具 {name}")     # 白名单执行：模型不能凭空造工具名
    return REGISTRY[name](arguments)


KV.delete("pred:PUMP-03")                     # 清掉上一格演示留下的缓存，保证看到 MISS
first, second = model_predict("PUMP-03"), model_predict("PUMP-03")
print("第一次:", json.dumps(first, ensure_ascii=False))
print("第二次:", json.dumps(second, ensure_ascii=False))
print(f"缓存对照: {first['cache']} → {second['cache']}")
assert (first["cache"], second["cache"]) == ("MISS", "HIT")
assert second["pred"] == first["pred"], "命中缓存却返回了不同结果 = 脏数据事故"


第一次: {"device": "PUMP-03", "pred": {"label": 1, "proba_wear": 1.0, "feats": {"samples": 600, "avg_vib": 5.095, "max_vib": 5.36, "slope": 0.0217, "last_vib": 5.207}}, "cache": "MISS"}
第二次: {"device": "PUMP-03", "pred": {"label": 1, "proba_wear": 1.0, "feats": {"samples": 600, "avg_vib": 5.095, "max_vib": 5.36, "slope": 0.0217, "last_vib": 5.207}}, "cache": "HIT"}
缓存对照: MISS → HIT


### Starter 14：LLM 客户端（离线规则引擎 ⇄ 真实服务，同一返回形状）
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

返回统一成 OpenAI `chat.completions` 的形状，所以**换后端不用改 ReAct 循环**——设 `P77_LLM_URL` 即切真实服务。
工单思政点落在这里：**企业数据不出域**。课堂默认全离线规则引擎，接真实大模型时先脱敏再出网。
`_plan()` 是这台"侦探"的办案直觉：**台账永远是无条件第一步**（先确认设备真实存在才谈其他），
再按问题关键词加时序 / 影响路径 / 相似案例 / 模型诊断——五步取证的"五步"就定义在这一格。

In [33]:
# -*- coding: utf-8 -*-
"""LLM 客户端：离线规则引擎（默认） ⇄ 真实 OpenAI 兼容服务（设 P77_LLM_URL 即切）

返回统一成 OpenAI chat.completions 形状，所以换后端不用改 ReAct 循环。
工单思政点：**企业数据不出域** —— 默认全离线；接真实大模型时先做脱敏再出网。
本机课堂既没有 GPU 也没有外网，规则引擎照样让 ReAct 四步循环完整跑通，
这是"教学可复现"与"生产接大模型"之间最省事的过渡桥。
"""
import re


def chat(messages: list, tools: list = None) -> dict:
    if os.environ.get("P77_LLM_URL"):                 # 生产分支（课堂不触发）
        return _chat_remote(messages, tools)
    return _chat_local(messages, tools)


def _chat_remote(messages, tools):
    import urllib.request
    body = json.dumps({"model": os.environ.get("P77_LLM_MODEL", "gpt-35-turbo"),
                       "messages": messages, "tools": tools or [], "temperature": 0.2}).encode()
    req = urllib.request.Request(os.environ["P77_LLM_URL"].rstrip("/") + "/chat/completions",
                                 data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=20) as r:
        return json.loads(r.read())


def _plan(q: str) -> list:
    """取证顺序：台账→时序→影响路径→案例→模型。

    台账是**无条件第一步**（值班工程师必须先确认"这是哪台设备、挂在哪条线"），
    其余四步按问题关键词裁剪——所以同一句话问"趋势"就不会白跑一次图查询。
    """
    p = ["query_relational"]
    if re.search(r"振动|趋势|最近|时序|窗口|爬升", q):
        p.append("query_timeseries")
    if re.search(r"影响|波及|联动|停产|停线|路径", q):
        p.append("impact_path")
    if re.search(r"案例|经验|类似|以前|怎么处理", q):
        p.append("semantic_search")
    p.append("model_predict")
    return p


def _used(messages) -> list:
    """从历史里看已经调用过哪些工具 → 保证 ReAct 不重复取证（等价 done_set）"""
    out = []
    for m in messages:
        c = m.get("content", "")
        if isinstance(c, str) and c.startswith("调用工具 "):
            out.append(c[5:])
    return out


def _bind_args(name, q, spec) -> str:
    m = re.search(r"PUMP-\d+|FAN-\d+|MOTOR-[A-Z0-9-]+|CNC-[A-Z0-9-]+", q)
    dev = m.group(0) if m else "PUMP-03"
    props = spec["function"]["parameters"]["properties"]
    return json.dumps({k: (dev if k == "device" else
                           q if k == "query" else 10) for k in props}, ensure_ascii=False)


def _chat_local(messages, tools):
    q = next(m["content"] for m in messages
             if m.get("role") == "user" and not str(m["content"]).startswith("工具"))
    used = _used(messages)
    nxt = next((t for t in _plan(q) if t not in used), None)
    if nxt and tools:
        spec = next(t for t in tools if t["function"]["name"] == nxt)
        return {"choices": [{"message": {
            "role": "assistant", "content": f"调用工具 {nxt}",
            "tool_calls": [{"id": f"call_{nxt}", "type": "function",
                            "function": {"name": nxt, "arguments": _bind_args(nxt, q, spec)}}]}}]}
    return {"choices": [{"message": {"role": "assistant",
                                     "content": "已收集全部证据，输出诊断工单（见结构化结论）。"}}]}


print("[LLM] 后端 = 离线规则引擎（P77_LLM_URL 未设置）")
_demo = chat([{"role": "system", "content": "…"},
              {"role": "user", "content": "PUMP-03振动持续变大，会影响哪些产线？"}], TOOLS_SPEC)
print("单步样例 →", json.dumps(_demo["choices"][0]["message"], ensure_ascii=False)[:220])


[LLM] 后端 = 离线规则引擎（P77_LLM_URL 未设置）
单步样例 → {"role": "assistant", "content": "调用工具 query_relational", "tool_calls": [{"id": "call_query_relational", "type": "function", "function": {"name": "query_relational", "arguments": "{\"device\": \"PUMP-03\"}"}}]}


### Starter 15：`def investigate(question)` —— **TODO-P7-5 ReAct 取证循环【B层】**
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

ReAct 四步 = **想（chat）→ 判（有没有 tool_calls）→ 动（run_tool）→ 看（观察写回 messages）**，
`MAX_STEPS=8` 防死循环，`trace` 逐步留痕（thought/act/args/obs）——工单的「有据可查」就是这一个列表。
两个坑都写在 docstring 里：
① `obs` 必须存**字符串**，因为 `_conclude()` 按 `json.loads(t["obs"])` 取用（已实现，别改契约）；
② **不把观察结果追加进 messages，规则引擎会永远重复同一个工具直到撞满 8 步**——这是学生版第一大红字来源。
`_conclude()` 已经把等级阈值（>0.7 报警 / >0.4 预警）、建议动作、波及面写好，你只补循环体。

In [35]:
# -*- coding: utf-8 -*-
"""运维智能体（ReAct 循环）：想→动手→看线索→结案

角色：两江智造"值班工程师小郑"。收到问题 → 依次取证（台账/时序/图谱/案例/模型）
→ 汇总结构化诊断工单。上限 8 步防死循环；**每一步都留痕**（工单思政：有据可查）。
"""
SYSTEM = ("你是两江智造集团设备智能运维中心的值班工程师。"
          "收到设备问题后按需调用工具取证：台账→时序→影响路径→相似案例→模型诊断，"
          "最后输出结论。不编造数据，只用工具返回的证据。")
MAX_STEPS = 8


def investigate(question: str) -> dict:
    """【P7-5 完成态】ReAct 四步循环（想→动→看→结）"""
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    trace = []
    for step in range(1, MAX_STEPS + 1):
        reply = chat(messages, TOOLS_SPEC)                     # ① 想：模型给下一步
        msg = reply["choices"][0]["message"]
        calls = msg.get("tool_calls") or []
        if not calls:                                          # ④ 结：没有工具调用即结案
            messages.append(msg)
            break
        call = calls[0]["function"]
        name, args = call["name"], json.loads(call["arguments"])
        obs = run_tool(name, args)                             # ② 动手：白名单执行工具
        obs_text = json.dumps(obs, ensure_ascii=False)
        messages.append({"role": "assistant", "content": f"调用工具 {name}"})
        messages.append({"role": "user", "content": f"工具{name}返回：{obs_text}"})  # ③ 看线索
        trace.append({"step": step, "thought": msg.get("content", ""),
                      "act": name, "args": args, "obs": obs_text})
    return {"question": question, "steps": len(trace), "trace": trace,
            "conclusion": _conclude(trace)}


def _conclude(trace) -> dict:
    """从取证轨迹提炼结论（等级/依据/建议动作/波及面）——结论只能引用 trace 里的证据"""
    obs = {t["act"]: t for t in trace if "obs" in t}
    pred = json.loads(obs["model_predict"]["obs"]) if "model_predict" in obs else {}
    wear = pred.get("pred", {}).get("proba_wear")
    cases = json.loads(obs["semantic_search"]["obs"]) if "semantic_search" in obs else []
    path = json.loads(obs["impact_path"]["obs"]) if "impact_path" in obs else {}
    level = "报警" if (wear or 0) > 0.7 else "预警" if (wear or 0) > 0.4 else "正常"
    return {"level": level, "prob_wear": wear,
            "suggest_action": cases[0]["meta"]["action"] if cases else "加密监测",
            "similar_case": cases[0]["id"] if cases else None,
            "evidence_keys": sorted(obs),
            "collateral_devices": [c["downstream"] for c in path.get("collateral", [])],
            "affected_lines": path.get("affected_lines", [])}


_r = investigate("PUMP-03振动持续变大，会影响哪些产线？以前怎么处理的？")
print("取证链路：")
for t in _r["trace"]:
    print(f"  第{t['step']}步 {t['act']:<16} 入参{json.dumps(t['args'], ensure_ascii=False)}"
          f" → 观察{len(t['obs'])}字符")
print("结论:", json.dumps(_r["conclusion"], ensure_ascii=False, indent=1))
assert _r["steps"] == 5, f"应五步取证，实际 {_r['steps']} 步"
assert set(_r["conclusion"]["evidence_keys"]) == set(REGISTRY), "五库取证有缺口"
assert _r["conclusion"]["suggest_action"] == "更换轴承", "相似案例召回错了处置动作"
assert "MOTOR-A2-1" in _r["conclusion"]["collateral_devices"]
print("✔ B层智能体闸：五步取证全覆盖 + 处置动作有据可查")


取证链路：
  第1步 query_relational 入参{"device": "PUMP-03"} → 观察465字符
  第2步 query_timeseries 入参{"device": "PUMP-03", "minutes": 10} → 观察87字符
  第3步 impact_path      入参{"device": "PUMP-03"} → 观察286字符
  第4步 semantic_search  入参{"query": "PUMP-03振动持续变大，会影响哪些产线？以前怎么处理的？", "device": "PUMP-03"} → 观察272字符
  第5步 model_predict    入参{"device": "PUMP-03"} → 观察176字符
结论: {
 "level": "报警",
 "prob_wear": 1.0,
 "suggest_action": "更换轴承",
 "similar_case": "case-001",
 "evidence_keys": [
  "impact_path",
  "model_predict",
  "query_relational",
  "query_timeseries",
  "semantic_search"
 ],
 "collateral_devices": [
  "MOTOR-A2-1",
  "CNC-A2-1"
 ],
 "affected_lines": [
  "LINE-A2"
 ]
}
✔ B层智能体闸：五步取证全覆盖 + 处置动作有据可查


### Starter 16：服务层 4 条路由（`http.server` 顶替 Litestar + Granian）
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

路由表驱动 = Litestar `@get/@post` 装饰器在做的事：一条 `(method, path前缀)` → 一个 handler。
四条验收路由 + 两条基准路由（`/bench/io` 睡 3ms 模拟下游往返、`/bench/cpu` 忙等 5ms 占着 GIL）都在这一格。
错误口径按工单速查表补齐：**坏 JSON 400 / 未知路由 404 / handler 异常 500**——服务不能因为一次查询崩。
两个容易忽略的压测细节：`protocol_version="HTTP/1.1"`（长连接，省反复握手）、
`request_queue_size=256`（**默认只有 5**，c=50 直接 connection reset——这就是"压测一上量就崩"的真凶）。
`serve_background(port=0)` 起后台服务，notebook 不阻塞、课堂不撞端口。

In [37]:
# -*- coding: utf-8 -*-
"""平台服务层：http.server(ThreadingHTTPServer) 顶替 Litestar + Granian

工单原厂 = Litestar(全异步) + Granian(Rust 运行时, --workers N)。本课堂按离线白名单
用标准库 http.server 起同样的 4 条路由，**接口契约(路径/字段/状态码)与工单验收口径一致**，
所以 /health、/api/device/{code}、/api/predict、/api/agent/ask 的验收逻辑一行不用改。
路由表驱动（= Litestar 的 @get/@post 装饰器在做的事）：一条 path 前缀 → 一个处理函数。
"""
from http.server import BaseHTTPRequestHandler, HTTPServer, ThreadingHTTPServer

ROUTES = {}


def route(method, prefix):
    def _wrap(fn):
        ROUTES[(method, prefix)] = fn
        return fn
    return _wrap


@route("GET", "/health")
def _h_health(handler, query, body):
    return 200, {"ok": True, "kv": KV.name, "svc": "smart-ops", "ver": "1.0",
                 "model_f1": MODEL_F1}


@route("GET", "/api/device")
def _h_device(handler, query, body):
    code = query["path"][len("/api/device"):].strip("/") or "PUMP-03"
    ck = f"dev:{code}"
    cached = KV.get(ck)
    if cached:                                             # 大屏 5s 轮询 → 命中缓存直接回
        return 200, {"source": "cache", "data": json.loads(cached)}
    data = {**device_profile(code), "ts_feats": window_feats(code)}
    KV.put(ck, json.dumps(data, ensure_ascii=False), ttl=CACHE_TTL_DEV)
    return 200, {"source": "live", "data": data}


@route("POST", "/api/predict")
def _h_predict(handler, query, body):
    return 200, model_predict(body.get("device", "PUMP-03"))


@route("POST", "/api/agent/ask")
def _h_ask(handler, query, body):
    who = (handler.headers.get("X-Client") or "anon")[:32]
    n = KV.incr(f"rl:{who}")                               # 限流计数（原子 INCR）
    if n > RATE_LIMIT_PER_HOUR:
        return 429, {"error": "rate limited", "n": n, "limit": RATE_LIMIT_PER_HOUR}
    r = investigate(body.get("question", ""))
    r["rate_used"] = n
    return 200, r


@route("GET", "/bench/cpu")
def _h_cpu(handler, query, body):
    """基准专用·计算密集：固定 5ms 忙等（占着 GIL 算）→ 证明"异步不加速 CPU"（B02 误区）"""
    t0 = time.perf_counter()
    acc = 0.0
    while time.perf_counter() - t0 < 0.005:
        acc += 0.1
    return 200, {"kind": "cpu", "busy_ms": 5, "acc": round(acc, 1)}


@route("GET", "/bench/io")
def _h_io(handler, query, body):
    """基准专用·IO 等待：固定 3ms sleep 模拟一次下游查库/网络往返 → 异步红利来源"""
    time.sleep(0.003)
    return 200, {"kind": "io", "wait_ms": 3}


class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"                          # 长连接：压测下省掉反复握手

    def _send(self, code, payload):
        data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def _dispatch(self, method):
        raw = body = None
        if method == "POST":
            length = int(self.headers.get("Content-Length") or 0)
            try:
                body = json.loads(self.rfile.read(length) or b"{}")
            except json.JSONDecodeError:
                self._send(400, {"error": "bad json"})     # 工单速查：422/400 一律回形状统一
                return
            raw = body
        path = self.path.split("?")[0]
        fn = next((f for (m, p), f in ROUTES.items()
                   if m == method and (path == p or path.startswith(p + "/"))), None)
        if fn is None:
            return self._send(404, {"error": "no route", "path": path})
        try:
            code, payload = fn(self, {"path": path}, raw or {})
            self._send(code, payload)
        except Exception as e:                              # 兜底 500：服务不能因为一次查询崩
            self._send(500, {"error": f"{type(e).__name__}: {e}"})

    def do_GET(self):
        self._dispatch("GET")

    def do_POST(self):
        self._dispatch("POST")

    def log_message(self, *a):                              # 压测时静音，否则 stdout 会被刷爆
        pass


class SmartOpsServer(ThreadingHTTPServer):
    daemon_threads = True              # 请求线程随进程退出，shutdown 不会被拖住
    request_queue_size = 256           # 默认只有 5！压测 c=50 时会直接 connection reset


class SingleThreadServer(HTTPServer):
    """对照组：单线程排队服务（等价 granian --workers 1 且不开并发）"""
    request_queue_size = 256


def serve_background(threaded=True, port=0):
    """起服务并立刻返回（notebook 里不能阻塞）；port=0 让内核挑空闲端口，避免撞车"""
    cls = SmartOpsServer if threaded else SingleThreadServer
    srv = cls((SERVICE_HOST, port), Handler)
    th = threading.Thread(target=srv.serve_forever, daemon=True)
    th.start()
    return srv, f"http://{SERVICE_HOST}:{srv.server_address[1]}"


SRV, BASE_URL = serve_background()
print(f"[服务] 已后台启动 → {BASE_URL} | 路由 {len(ROUTES)} 条 | 模式 ThreadingHTTPServer")
print("       (真实上线换 granian --interface asgi service.app:app --port 8900 --workers 4)")


[服务] 已后台启动 → http://127.0.0.1:57493 | 路由 6 条 | 模式 ThreadingHTTPServer
       (真实上线换 granian --interface asgi service.app:app --port 8900 --workers 4)


### Starter 17：D层验收——四个 API 全 200 + live→cache 差异**逐字留痕**
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

D 层达标口径「4个API全部返回200，截图 live→cache 差异」在这里落地：
`http()` 是 urllib 写的极简 curl（连 429/404 也要把响应体读回来）。
调之前先 `KV.delete()`，否则缓存早就 HIT 了，**live→cache 这一对根本观察不到**——
这是本课最典型的"验收要看得到差异，而不是只看绿灯"。
工单要截图，notebook 给不了 → 改成把 5 次响应体写进 `w09_scratch/reports/D_layer_live_vs_cache.json`，
**留痕比截图更可查**（答辩当场打开对读，还能 diff）。

In [39]:
# -*- coding: utf-8 -*-
"""D层·学徒验收：bash deploy/run.sh 之后 curl 四个 API —— 这里用 urllib 等价发起
达标标准：4 个 API 全部 200 + 观察 live→cache 两次响应差异

工单要求"截图留痕"。notebook 里没法截图，就把两次响应体逐字落盘到
`w09_scratch/reports/D_layer_live_vs_cache.json`，答辩时打开对读——**留痕比截图更可查**。
"""
import urllib.error
import urllib.request


def http(method, path, payload=None, headers=None, base=None):
    """极简 REST 客户端（等价 curl -s -w '%{http_code}'）"""
    data = json.dumps(payload).encode() if payload is not None else None
    req = urllib.request.Request((base or BASE_URL) + path, data=data, method=method,
                                 headers={"Content-Type": "application/json", **(headers or {})})
    try:
        with urllib.request.urlopen(req, timeout=15) as r:
            return r.status, json.loads(r.read() or b"{}")
    except urllib.error.HTTPError as e:                     # 429/404 也要拿到响应体
        return e.code, json.loads(e.read() or b"{}")


KV.delete("dev:PUMP-03")                                    # 保证第一次一定是 live
KV.delete("pred:PUMP-03")                                   # 保证推理是 MISS→HIT 对照
CALLS = [("GET", "/health", None),
         ("GET", "/api/device/PUMP-03", None),              # 第一次 → live
         ("GET", "/api/device/PUMP-03", None),              # 第二次 → cache
         ("POST", "/api/predict", {"device": "PUMP-03"}),
         ("POST", "/api/agent/ask", {"question": "PUMP-03振动持续变大，影响哪些产线？以前怎么处理？"})]
log = []
for m, p, body in CALLS:
    code, js = http(m, p, body)
    log.append({"method": m, "path": p, "status": code, "body": js})
    brief = json.dumps(js, ensure_ascii=False)
    print(f"{code} {m:<4} {p:<26} {brief[:110]}{'…' if len(brief) > 110 else ''}")

statuses = [r["status"] for r in log]
assert statuses == [200] * 5, f"D层闸：4个API须全 200，实际 {statuses}"
src1 = log[1]["body"]["data"]["device"]["code"] if log[1]["body"]["data"]["found"] else "?"
assert log[1]["body"]["source"] == "live" and log[2]["body"]["source"] == "cache", "缓存链路没生效"
assert log[1]["body"]["data"] == log[2]["body"]["data"], "live 与 cache 内容不一致 = 脏缓存"
assert log[3]["body"]["cache"] == "MISS", "上一格已清缓存，此处应为 MISS"
assert log[3]["body"]["pred"]["label"] == 1, \
    "PUMP-03 应被判为磨损(1)；拿到 0 说明窗口特征是空的 → 先补 TODO-P7-2"
assert len(log[4]["body"]["trace"]) == 5, \
    f"智能体五步取证，实际 {len(log[4]['body']['trace'])} 步 → 先补 TODO-P7-5"
with open(os.path.join(REPORTS, "D_layer_live_vs_cache.json"), "w", encoding="utf-8") as f:
    json.dump(log, f, ensure_ascii=False, indent=1)
print(f"✔ D层达标：全 200 | live→cache 同数据不同来源 | proba_wear="
      f"{log[3]['body']['pred']['proba_wear']} | 留痕 reports/D_layer_live_vs_cache.json")


200 GET  /health                    {"ok": true, "kv": "ttl-kv(离线兜底)", "svc": "smart-ops", "ver": "1.0", "model_f1": 0.95}
200 GET  /api/device/PUMP-03        {"source": "live", "data": {"found": true, "device": {"code": "PUMP-03", "name": "3号泵站循环泵", "line": "LINE-A2",…
200 GET  /api/device/PUMP-03        {"source": "cache", "data": {"found": true, "device": {"code": "PUMP-03", "name": "3号泵站循环泵", "line": "LINE-A2"…
200 POST /api/predict               {"device": "PUMP-03", "pred": {"label": 1, "proba_wear": 1.0, "feats": {"samples": 600, "avg_vib": 5.095, "max…
200 POST /api/agent/ask             {"question": "PUMP-03振动持续变大，影响哪些产线？以前怎么处理？", "steps": 5, "trace": [{"step": 1, "thought": "调用工具 query_relation…
✔ D层达标：全 200 | live→cache 同数据不同来源 | proba_wear=1.0 | 留痕 reports/D_layer_live_vs_cache.json


### Starter 18：B层复测——问题换成 FAN-02 再测一轮（工单 B 层步骤3）
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

同一个 `/api/agent/ask`，只换设备名：爬升泵判『报警』、平稳风机判『正常』。
**换设备不改一行代码还自洽 = 平台真在读数据，不是在背答案。**
这一格也是 B 层最容易失分的地方：只测 PUMP-03 就签字，答辩老师随手换个设备名就露馅。

In [41]:
# -*- coding: utf-8 -*-
"""B层·步骤3 复测：把 agent/ask 的问题从 PUMP-03 换成 FAN-02 —— 同一套代码要给出**相反**的结论

工单B层的达标线不是"能跑"，而是"换个设备还自洽"：
  PUMP-03 振动在爬升 → 报警 / 建议更换轴承 / 波及 LINE-A2
  FAN-02  振动平稳   → 正常 / 加密监测 / 无下游波及（风机不给别的设备供电）
换设备不改一行代码 = 平台真的在**读数据**，而不是在背答案。这是答辩时最容易戳穿的一点。
"""
RETEST = {}
for dev, exp_level in (("PUMP-03", "报警"), ("FAN-02", "正常")):
    code, js = http("POST", "/api/agent/ask",
                    {"question": f"{dev}振动趋势如何，会影响哪些产线？以前怎么处理？"})
    concl = js["conclusion"]
    RETEST[dev] = {"status": code, "steps": js["steps"], **concl}
    print(f"{dev:<9} {code} | {js['steps']}步取证 | 等级={concl['level']} proba_wear={concl['prob_wear']} "
          f"| 动作={concl['suggest_action']} | 波及线={concl['affected_lines']}")
    assert code == 200 and js["steps"] == 5, f"五步取证没跑满（{dev}）→ 先补 TODO-P7-5"
    assert concl["level"] == exp_level, \
        f"{dev} 结论应为『{exp_level}』，实际『{concl['level']}』——结论必须跟着数据走，不是跟着模板走"
print("✔ B层复测闸：同一套代码对爬升泵判『报警』、对平稳风机判『正常』，读的是数据不是答案")


PUMP-03   200 | 5步取证 | 等级=报警 proba_wear=1.0 | 动作=更换轴承 | 波及线=['LINE-A2']
FAN-02    200 | 5步取证 | 等级=正常 proba_wear=0.0 | 动作=紧固螺栓 | 波及线=['LINE-B1']
✔ B层复测闸：同一套代码对爬升泵判『报警』、对平稳风机判『正常』，读的是数据不是答案


### Starter 19：端到端总检 `verify()`（B05）——断言全绿才签字放行
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

等价 starter 的 `verify.sh`：五库段 / 模型段 / 智能体段 / 服务段 / 限流段，逐项打 `[PASS]/[FAIL]`，
最后 `assert` 总数——**一环红全线返工，不允许"我本地是好的"**。
限流段先 `KV.delete(rl:who)` 再测，所以这一格**可重复运行**（幂等），不会因为跑了两遍就误红。
C 层看「五库段全绿」，B 层看「ALL PASS」——两级闸门在同一个函数里，靠 section 名字区分。

In [43]:
# -*- coding: utf-8 -*-
"""端到端总检 verify()：等价 starter 的 verify.sh —— 断言全绿才签字放行（B05）

C层闸 = 五库段全绿；B层闸 = ALL PASS。每一项都是**真跑真断言**，不是打印"应该通过"。
"""


def verify():
    checks = []

    def chk(section, name, ok, detail=""):
        checks.append((section, name, bool(ok), detail))

    # ---- 五库段（C层验收口径）----
    prof = device_profile("PUMP-03")
    chk("五库", "关系SQLite台账", prof["found"] and prof["point_count"] == 3 and prof["alert_count"] == 2,
        f"测点{prof['point_count']} 告警{prof['alert_count']}")
    wf = window_feats("PUMP-03")
    chk("五库", "时序窗口特征", wf["samples"] > 300 and wf["slope"] > 0,
        f"n={wf['samples']} avg={wf['avg_vib']} slope={wf['slope']}")
    ip = impact_path("PUMP-03")
    chk("五库", "图影响路径", set(c["downstream"] for c in ip["collateral"]) ==
        closure_reachable("PUMP-03", "上游供电") and ip["affected_lines"],
        f"下游{ip['downstream_count']}个 波及线{ip['affected_lines']}")
    hits = search("泵振动持续变大怎么办", 2, device="PUMP-03")      # 智能体的真实调用形态
    raw = search("泵振动持续变大怎么办", 2)                          # 纯语义（会被长文档偏置带偏）
    _top1 = (hits or [{}])[0].get("id", "无（TODO-P7-4 未填）")
    _raw1 = (raw or [{}])[0].get("id", "无")
    chk("五库", "向量案例召回", bool(hits) and hits[0]["id"] == "case-001",
        f"过滤后Top1={_top1} | 纯语义Top1={_raw1}（长度偏置→靠过滤修正）")
    KV.delete("verify:k")
    KV.put("verify:k", "v", ttl=30)
    chk("五库", "KV缓存读写+TTL", KV.get("verify:k") == "v" and KV.ttl_left("verify:k") > 0,
        f"剩余TTL={KV.ttl_left('verify:k')}")

    # ---- 模型段（P6 抽查线）----
    chk("模型", "F1(macro)≥0.80", MODEL_F1 >= 0.80, f"F1={MODEL_F1}")
    pump = model_predict("PUMP-03")["pred"]
    fan = model_predict("FAN-02")["pred"]
    chk("模型", "爬升泵判磨损/平稳风机判正常", pump["label"] == 1 and fan["label"] == 0,
        f"PUMP-03 proba={pump['proba_wear']} FAN-02 proba={fan['proba_wear']}")

    # ---- 智能体段（B层 ALL PASS 口径）----
    ag = investigate("FAN-02振动趋势如何，会影响哪些产线？以前怎么处理？")
    chk("智能体", "五步取证全覆盖", ag["steps"] == 5 and
        set(ag["conclusion"]["evidence_keys"]) == set(REGISTRY), f"steps={ag['steps']}")
    chk("智能体", "每步留痕可追溯", all(t["thought"] and t["obs"] for t in ag["trace"]),
        "thought/act/args/obs 齐全")
    chk("智能体", "换设备(FAN-02)结论自洽", ag["conclusion"]["level"] == "正常" and
        ag["conclusion"]["collateral_devices"] == [], f"level={ag['conclusion']['level']}")

    # ---- 服务段（D层 4 API）----
    KV.delete("dev:FAN-02")
    for name, code, js in [("GET /health", *http("GET", "/health")),
                           ("GET /api/device/FAN-02", *http("GET", "/api/device/FAN-02")),
                           ("POST /api/predict", *http("POST", "/api/predict", {"device": "FAN-02"})),
                           ("POST /api/agent/ask", *http("POST", "/api/agent/ask",
                                                         {"question": "FAN-02振动趋势如何？"}))]:
        chk("服务", name + " → 200", code == 200, f"status={code}")
    chk("服务", "device 缓存 live→cache",
        http("GET", "/api/device/FAN-02")[1]["source"] == "cache")
    bad = http("GET", "/api/nothing")[0]
    chk("服务", "未知路由 404 而非崩溃", bad == 404, f"status={bad}")

    # ---- 限流段 ----
    who = "rl-verify-%d" % int(_NOW[0])
    KV.delete(f"rl:{who}")            # 总检可重复运行：先把待测计数清零（幂等，别把上一轮余量算进来）
    codes = [http("POST", "/api/agent/ask", {"question": "FAN-02"}, {"X-Client": who})[0]
             for _ in range(RATE_LIMIT_PER_HOUR + 3)]
    chk("限流", f"前{RATE_LIMIT_PER_HOUR}次放行/超出即429",
        codes[:RATE_LIMIT_PER_HOUR] == [200] * RATE_LIMIT_PER_HOUR
        and set(codes[RATE_LIMIT_PER_HOUR:]) == {429},
        f"200×{codes.count(200)} 429×{codes.count(429)}")

    # ---- 报告 ----
    for section, name, ok, detail in checks:
        print(f"[{'PASS' if ok else 'FAIL'}] {section:<4} {name:<26} {detail}")
    n_ok = sum(1 for c in checks if c[2])
    print(f"\n{'ALL PASS ✅' if n_ok == len(checks) else '有红灯，禁止上线 ❌'}"
          f"  →  {n_ok}/{len(checks)} checks green")
    assert n_ok == len(checks), [c for c in checks if not c[2]]
    return checks


CHECKS = verify()


[PASS] 五库   关系SQLite台账                 测点3 告警2
[PASS] 五库   时序窗口特征                     n=600 avg=5.095 slope=0.0217
[PASS] 五库   图影响路径                      下游2个 波及线['LINE-A2']
[PASS] 五库   向量案例召回                     过滤后Top1=case-001 | 纯语义Top1=case-003（长度偏置→靠过滤修正）
[PASS] 五库   KV缓存读写+TTL                 剩余TTL=30.0
[PASS] 模型   F1(macro)≥0.80             F1=0.95
[PASS] 模型   爬升泵判磨损/平稳风机判正常             PUMP-03 proba=1.0 FAN-02 proba=0.0
[PASS] 智能体  五步取证全覆盖                    steps=5
[PASS] 智能体  每步留痕可追溯                    thought/act/args/obs 齐全
[PASS] 智能体  换设备(FAN-02)结论自洽            level=正常
[PASS] 服务   GET /health → 200          status=200
[PASS] 服务   GET /api/device/FAN-02 → 200 status=200
[PASS] 服务   POST /api/predict → 200    status=200
[PASS] 服务   POST /api/agent/ask → 200  status=200
[PASS] 服务   device 缓存 live→cache       
[PASS] 服务   未知路由 404 而非崩溃              status=404
[PASS] 限流   前60次放行/超出即429              200×60 429×3

ALL PASS ✅  →  17/17 checks green


### Starter 20：A层·拓展1 检索选型——三套方案用**同一把尺子**量 recall@1
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

工单原厂要求"哈希嵌入换 bge-small-zh 对比召回"。**bge 权重要联网下载，本机不可测 → 本格不给它编数字**，
只做三套离线可复现方案的对照：A 哈希词袋 / B 字符 n-gram TF-IDF（sklearn 本地训练）/ C 哈希 + 设备元数据过滤。
6 条人工标注的评测集 `QREL` 才是这一格的主角：**没有评测集，选型就是拍脑袋**。
结论常常反直觉——4 条案例的超小语料上 TF-IDF 的 IDF 权重会失真，反而不如哈希。
这正是"选型要用评测集量，不能靠名气拍板"的最好教材，也是决策矩阵第 4 行的实测来源。

In [45]:
# -*- coding: utf-8 -*-
"""A层·拓展1 —— 嵌入/检索选型对比：三套方案同一把尺子量召回率（全部实测）

工单原厂要求"哈希嵌入换 bge-small-zh 对比召回"。bge 权重需联网下载，课堂离线不可测，
**所以本格不给 bge 编数字**，只给三套离线可复现方案的实测对照：
  方案A 哈希词袋(字+bigram×2) 纯语义排序
  方案B 字符 n-gram TF-IDF（sklearn 本地训练，不需下载）
  方案C 哈希词袋 + 设备元数据过滤（先缩货架再排序 = Chroma 的 where）
结论常常反直觉：4 条案例的超小语料上 TF-IDF 的 IDF 权重会失真，反而不如哈希——
这就是"选型必须用评测集量，不能靠名气拍板"的最好教材。
"""
from sklearn.feature_extraction.text import TfidfVectorizer

CORPUS = [c["text"] for c in CASES]
_tv = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3), sublinear_tf=True)
_tv.fit(CORPUS)
_mat_tv = _tv.transform(CORPUS).toarray()
_mat_tv = _mat_tv / np.linalg.norm(_mat_tv, axis=1, keepdims=True)
_mat_hash = np.vstack([embed_hash(c["text"]) for c in CASES])


def embed_tfidf(text: str) -> np.ndarray:
    v = _tv.transform([text]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return (v / np.where(n == 0, 1.0, n))[0]


def topk(query, mat, embed, k=3, device=None) -> list:
    sims = mat @ embed(query)
    full = list(np.argsort(-sims))
    if device:                                             # 元数据过滤 = Chroma where={"device":…}
        order = [i for i in full if CASES[i]["meta"]["device"] == device]
        order += [i for i in full if i not in order]        # 不足则回退全库补齐
    else:
        order = full
    return [CASES[i]["id"] for i in order[:k]]


def evaluate(name, mat, embed, use_device=False, k=1) -> dict:
    hit, rows = 0, []
    for q, truth, dev in QREL:
        got = topk(q, mat, embed, k=3, device=dev if use_device else None)
        ok = truth in got[:k]
        hit += ok
        rows.append(f"   {'✓' if ok else '✗'} {q:<14} 期望{truth} 实得{got[:k]}")
    print(f"—— 方案{name}（{'含设备过滤' if use_device else '纯语义'}）召回明细：")
    print("\n".join(rows))
    return {"scheme": name, "recall@1": round(hit / len(QREL), 3),
            "hits": hit, "total": len(QREL)}


def bench_embed_ms(embed, reps=200):
    t0 = time.perf_counter()
    for _ in range(reps):
        embed("泵振动持续变大如何处理")
    return round((time.perf_counter() - t0) / reps * 1000, 3)


R_HASH = evaluate("A 哈希词袋", _mat_hash, embed_hash)
R_TFIDF = evaluate("B 字符TF-IDF", _mat_tv, embed_tfidf)
R_FILTER = evaluate("C 哈希+元数据过滤", _mat_hash, embed_hash, use_device=True)
for r in (R_HASH, R_TFIDF, R_FILTER):
    r["emb_ms"] = bench_embed_ms(embed_tfidf if r["scheme"].startswith("B") else embed_hash)

print("\n召回@1 对照: " + " | ".join(f"{r['scheme']}={r['recall@1']}" for r in (R_HASH, R_TFIDF, R_FILTER)))
print(f"单条嵌入耗时: 哈希{R_HASH['emb_ms']}ms / TF-IDF{R_TFIDF['emb_ms']}ms")
CHOSEN = R_FILTER
assert CHOSEN["recall@1"] >= 0.80, f"上线召回下限未达：{CHOSEN}"
assert CHOSEN["recall@1"] > R_HASH["recall@1"], "元数据过滤应带来可测量的召回增益，否则说明评测集太偏"
print(f"✔ A层召回闸：选定方案 = {CHOSEN['scheme']}，recall@1 = {CHOSEN['recall@1']}（≥0.80）")
print("  ⚠ 诚实声明：bge-small-zh 需联网下载权重，本机未测 → 不给它编造召回数字；"
      "上真语义模型时接口不变，把 embed_hash 换掉即可。")


—— 方案A 哈希词袋（纯语义）召回明细：
   ✗ 泵振动持续变大如何处理    期望case-001 实得['case-003']
   ✓ 风机振动超标怎么办      期望case-003 实得['case-003']
   ✓ 叶轮堵了压力不稳       期望case-002 实得['case-002']
   ✓ 联轴器没对中电流异常     期望case-004 实得['case-004']
   ✓ 轴承磨损该换什么       期望case-001 实得['case-001']
   ✓ 振动大温度也高        期望case-003 实得['case-003']
—— 方案B 字符TF-IDF（纯语义）召回明细：
   ✗ 泵振动持续变大如何处理    期望case-001 实得['case-002']
   ✓ 风机振动超标怎么办      期望case-003 实得['case-003']
   ✓ 叶轮堵了压力不稳       期望case-002 实得['case-002']
   ✓ 联轴器没对中电流异常     期望case-004 实得['case-004']
   ✓ 轴承磨损该换什么       期望case-001 实得['case-001']
   ✓ 振动大温度也高        期望case-003 实得['case-003']
—— 方案C 哈希+元数据过滤（含设备过滤）召回明细：
   ✓ 泵振动持续变大如何处理    期望case-001 实得['case-001']
   ✓ 风机振动超标怎么办      期望case-003 实得['case-003']
   ✓ 叶轮堵了压力不稳       期望case-002 实得['case-002']
   ✓ 联轴器没对中电流异常     期望case-004 实得['case-004']
   ✓ 轴承磨损该换什么       期望case-001 实得['case-001']
   ✓ 振动大温度也高        期望case-003 实得['case-003']

召回@1 对照: A 哈希词袋=0.833 | B 字符TF-IDF=0.833 | C 哈希+元数据过滤=1.0
单条嵌入耗时: 哈希0.016ms / TF-IDF0.376ms
✔ A

### Starter 21：A层·拓展2 压测与 workers 调优——把 B02 的误区量出来
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

原厂口径是 `granian --workers N` + `ab -n 1000 -c 50`。本机只有 stdlib，于是把三件事量清楚：
① **并发红利从哪来**：`/bench/io`（睡 3ms，等待可重叠）单线程→多线程 QPS 真涨；
`/bench/cpu`（忙等 5ms，抢 GIL）几乎不涨——这一对就是「异步红利于 IO 等待，不救 CPU 密集」的实验证据；
② **线上端点并发梯度** c=1/10/50 + 冷启动 40 个设备码（缓存没挡住时的真实 p95，断言 <500ms）；
③ **工单口径外部 `ab`**：本机有 `ab` 就真跑并断言 `failed==0` 且 KV 命中数达标；没有就打印「该项未执行（不做假数据）」——
**绝不为了报告好看而编 QPS**。
本格还会坦白一件事：同进程 stdlib 客户端自己也吃 GIL，所以它的 QPS 上限是被**客户端**卡住的——
**压测工具也是被测系统的一部分**。

In [47]:
# -*- coding: utf-8 -*-
"""A层·拓展2 —— 压测与 workers 调优（数字全部本机实测，换机幅度会变、趋势可复用）

原厂口径是 `granian --interface asgi … --workers N` + `ab -n 1000 -c 50`。
课堂白名单只有标准库 http.server，于是把三件事量清楚：
  ① 并发红利从哪来 —— 有 IO 等待时，多线程重叠等待，QPS 真涨；
  ② 异步不是万能加速 —— CPU 密集端点受 GIL 限制，加并发几乎不涨（B02 误区实测）；
  ③ 压测工具自己也是被测系统 —— 客户端与服务器同进程时 QPS 上限被 GIL 卡住，
     所以工单口径要用外部 C 程序 ab 复测（有则跑，没有就如实标注）。
"""
from concurrent.futures import ThreadPoolExecutor


def bench(path, payload=None, concurrency=1, total=300, base=None):
    """stdlib 并发压测：返回 QPS 与 p50/p95/p99 延迟（真实墙钟，非估算）"""
    def _one(_):
        t0 = time.perf_counter()
        code, _js = http("POST" if payload else "GET", path, payload, base=base)
        return code, (time.perf_counter() - t0) * 1000
    t_start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        res = list(ex.map(_one, range(total)))
    wall_s = time.perf_counter() - t_start
    lat = sorted(r[1] for r in res)
    ok = sum(1 for r in res if r[0] == 200)
    return {"path": path, "c": concurrency, "n": total, "ok": ok,
            "qps": round(ok / max(wall_s, 1e-9), 1),
            "p50_ms": round(lat[len(lat) // 2], 2),
            "p95_ms": round(lat[int(len(lat) * 0.95) - 1], 2),
            "p99_ms": round(lat[int(len(lat) * 0.99) - 1], 2)}


print("① 线上热点端点 /api/device/PUMP-03（TTL 内基本命中 KV，正是大屏轮询的真实形态）")
KV.delete("dev:PUMP-03")
io_rows = [bench("/api/device/PUMP-03", concurrency=c, total={1: 100, 10: 200, 50: 500}[c])
           for c in (1, 10, 50)]
for r in io_rows:
    print(f"   c={r['c']:>2} n={r['n']:>4} QPS={r['qps']:>7} "
          f"p50={r['p50_ms']:>6}ms p95={r['p95_ms']:>6}ms p99={r['p99_ms']:>6}ms 200×{r['ok']}")


def bench_cold(n=40):
    """冷路径：每次都用没缓存过的设备号，逼出『关系库+时序窗口+写缓存』全流程耗时"""
    lat = []
    for i in range(n):
        t0 = time.perf_counter()
        http("GET", f"/api/device/COLD-{i:03d}")
        lat.append((time.perf_counter() - t0) * 1000)
    lat.sort()
    return {"n": n, "p50_ms": round(lat[n // 2], 2), "p95_ms": round(lat[int(n * 0.95) - 1], 2)}


cold = bench_cold()
print(f"   冷路径(全部回源) n={cold['n']} p50={cold['p50_ms']}ms p95={cold['p95_ms']}ms")
assert io_rows[2]["p95_ms"] < 500, f"热端点 p95 超标：{io_rows[2]['p95_ms']}ms"
assert cold["p95_ms"] < 500, f"冷路径 p95 超标：{cold['p95_ms']}ms（回源太慢，缓存救不回来）"
BENCH_COLD = cold

print("\n② 并发模型对照：同一份 Handler，只换 单线程排队 vs 多线程")
srv_one, url_one = serve_background(threaded=False, port=0)
MODEL_ROWS = {}
for kind, path in (("IO等待3ms", "/bench/io"), ("CPU忙等5ms", "/bench/cpu")):
    one = bench(path, concurrency=50, total=400, base=url_one)
    thr = bench(path, concurrency=50, total=400)
    gain = round(thr["qps"] / one["qps"], 2)
    MODEL_ROWS[kind] = {"single": one, "threaded": thr, "gain": gain}
    print(f"   {kind:<10} 单线程 QPS={one['qps']:>7} p95={one['p95_ms']:>7}ms | "
          f"多线程 QPS={thr['qps']:>7} p95={thr['p95_ms']:>6}ms | 提升 {gain}×")
srv_one.shutdown(); srv_one.server_close()                 # 对照服务用完即关

print("\n③ 结论（实测推出来的，不是背出来的）：")
IO_GAIN = MODEL_ROWS["IO等待3ms"]["gain"]
CPU_GAIN = MODEL_ROWS["CPU忙等5ms"]["gain"]
print(f"   有 IO 等待 → 多线程把等待重叠掉，QPS {IO_GAIN}×；")
print(f"   纯 CPU 计算 → GIL 下加并发没用，QPS {CPU_GAIN}×（要真提速得多进程 workers / Rust 运行时）")
assert IO_GAIN > 2, f"IO 等待型端点的并发红利未体现（{IO_GAIN}×）"
assert CPU_GAIN < IO_GAIN, "CPU 密集端点不该比 IO 端点更容易扩展——测出来反转就要怀疑测量方法"

print("\n④ 工单口径 ab -n 1000 -c 50（外部 C 压测客户端；本机没有就如实标注未跑）")
AB = {}
_ab = shutil.which("ab")
if _ab:
    hits_before = KV.hits
    p = subprocess.run([_ab, "-n", "1000", "-c", "50", BASE_URL + "/api/device/PUMP-03"],
                       capture_output=True, text=True, timeout=180)
    for line in (p.stdout or "").splitlines():
        k = line.split(":")[0].strip()
        if k in ("Requests per second", "Failed requests", "Complete requests",
                 "Time per request", "50%", "66%", "95%", "99%"):
            print("   ab>", line.strip())
        if k == "Requests per second":
            AB["qps"] = float(line.split(":")[1].split()[0])
        if k == "Failed requests":
            AB["failed"] = int(line.split(":")[1].split()[0])
        if k == "Complete requests":
            AB["complete"] = int(line.split(":")[1].split()[0])
    AB["note"] = ("外部客户端复测(不含本进程 GIL 限制)；打的是热点端点 /api/device，"
                  "TTL 内全部命中 KV → 这条 QPS 量的是『缓存挡住回源』之后的上限")
    AB["kv_hits"] = KV.hits - hits_before
    print(f"   ab 期间 KV 命中 {AB['kv_hits']} 次（缓存把 {AB['kv_hits']} 次回源挡掉了）")
    assert AB.get("failed", 0) == 0, f"ab 出现失败请求 {AB.get('failed')}，服务端不稳"
    assert AB["kv_hits"] >= 900, "大屏轮询应几乎全命中缓存，否则 TTL 设计失败"
else:
    AB["note"] = "本机 PATH 无 ab → 该项未执行（不做假数据），以①②的 stdlib 实测为准"
    print("   " + AB["note"])
BENCH_ROWS = {"online": io_rows, "model": MODEL_ROWS, "ab": AB,
              "gain_io": IO_GAIN, "gain_cpu": CPU_GAIN}
print("✔ A层压测闸：p95 达标 + 并发红利量化 + 计算密集瓶颈定位 + ab 口径留痕")


① 线上热点端点 /api/device/PUMP-03（TTL 内基本命中 KV，正是大屏轮询的真实形态）
   c= 1 n= 100 QPS= 1607.4 p50=  0.55ms p95=  0.68ms p99=  0.84ms 200×100
   c=10 n= 200 QPS= 1783.9 p50=  5.42ms p95=  6.44ms p99=  6.88ms 200×200
   c=50 n= 500 QPS= 1734.8 p50= 27.48ms p95= 29.24ms p99= 30.04ms 200×500
   冷路径(全部回源) n=40 p50=3.38ms p95=6.63ms

② 并发模型对照：同一份 Handler，只换 单线程排队 vs 多线程
   IO等待3ms    单线程 QPS=  198.1 p95= 290.23ms | 多线程 QPS= 1857.5 p95= 30.31ms | 提升 9.38×
   CPU忙等5ms   单线程 QPS=  179.3 p95= 299.37ms | 多线程 QPS=  177.8 p95= 290.6ms | 提升 0.99×

③ 结论（实测推出来的，不是背出来的）：
   有 IO 等待 → 多线程把等待重叠掉，QPS 9.38×；
   纯 CPU 计算 → GIL 下加并发没用，QPS 0.99×（要真提速得多进程 workers / Rust 运行时）

④ 工单口径 ab -n 1000 -c 50（外部 C 压测客户端；本机没有就如实标注未跑）
   ab> Complete requests:      1000
   ab> Failed requests:        0
   ab> Requests per second:    3993.39 [#/sec] (mean)
   ab> Time per request:       12.521 [ms] (mean)
   ab> Time per request:       0.250 [ms] (mean, across all concurrent requests)
   ab 期间 KV 命中 1000 次（缓存把 1000 次回源挡掉了）
✔ A层压测闸：p95

### Starter 22：A层·拓展3 上线交付三件套 + 选型/压测报告（**TODO-P7-A**）
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

生成 `deploy/run.sh`（可执行位都 assert）、`Dockerfile`、`docker-compose.yml`，并做**静态校验**
（关键键齐不齐、有没有 Tab 缩进——compose 一缩进错就全线崩）；`docker compose config` 只在守护进程可用时才真跑。
报告 `P7_A_选型与压测对比报告.md` 的决策矩阵由**你前面几格实测到的变量**直接填出，不留空话。
`CONCLUSION` 三句（存储/检索/服务选型）就是学生版要自己写的部分：**必须引用你自己测到的数字**，抄模板没分。

In [49]:
# -*- coding: utf-8 -*-
"""A层·拓展3 —— 上线交付物：deploy/run.sh + Dockerfile + docker-compose.yml + 选型/压测报告

课堂边界（诚实声明）：容器部署要 Docker 守护进程，机房实训机不保证有。
所以本格的达标口径是"交付物齐套 + 静态校验通过 + 报告有实测数字"，
`docker compose config` 只在守护进程可用时才真跑，不可用就如实标注"未执行"。
"""
RUN_SH = """#!/usr/bin/env bash
# P7 一键播种+启动（对应工单 D层 步骤1）
set -euo pipefail
cd "$(dirname "$0")/.."
PY=${PY:-/opt/anaconda3/envs/pp03/bin/python}
redis-cli -p 6399 ping >/dev/null 2>&1 || echo "[warn] 无 redis，KV 自动回退离线 TTL-KV"
"$PY" -c "import p77_smart_ops" 2>/dev/null || true
"$PY" deploy_entry.py seed      # 依次播种五库 + 训练模型
"$PY" deploy_entry.py verify    # 端到端断言，全绿才继续
exec "$PY" deploy_entry.py serve # granian --interface asgi service.app:app --port 8900 --workers 4
"""
DOCKERFILE = """FROM python:3.12-slim
WORKDIR /app
COPY requirements.lock .
RUN pip install --no-cache-dir -r requirements.lock
COPY . .
EXPOSE 8900
HEALTHCHECK CMD curl -fsS http://127.0.0.1:8900/health || exit 1
CMD ["granian", "--interface", "asgi", "service.app:app", "--host", "0.0.0.0", "--port", "8900", "--workers", "4"]
"""
COMPOSE = """services:
  web:
    build: .
    ports: ["8900:8900"]
    environment: {REDIS_URL: "redis://redis:6379/0"}
    depends_on: [redis]
    restart: unless-stopped
  redis:
    image: redis:7-alpine
    command: ["redis-server", "--port", "6379"]
    healthcheck: {test: ["CMD", "redis-cli", "ping"], interval: 10s}
"""
DEPLOY = os.path.join(SCRATCH, "deploy")
for name, content, mode in [("run.sh", RUN_SH, 0o755), ("Dockerfile", DOCKERFILE, 0o644),
                            ("docker-compose.yml", COMPOSE, 0o644)]:
    p = os.path.join(DEPLOY, name)
    with open(p, "w", encoding="utf-8") as f:
        f.write(content)
    os.chmod(p, mode)


def validate_compose(text: str) -> list:
    """极简静态校验（无 pyyaml 也能查出关键键，避免"缩进一错全线崩"）"""
    problems = []
    for need in ("services:", "  web:", "    build: .", 'ports: ["8900:8900"]', "  redis:"):
        if need not in text:
            problems.append(f"缺少 {need}")
    if "\t" in text:
        problems.append("compose 不允许 Tab 缩进")
    return problems


print("交付物 →", os.path.relpath(DEPLOY, BASE), "|", ", ".join(sorted(os.listdir(DEPLOY))))
PROBLEMS = validate_compose(COMPOSE)
print("compose 静态校验:", "通过 ✔" if not PROBLEMS else PROBLEMS)
assert not PROBLEMS, PROBLEMS
assert os.access(os.path.join(DEPLOY, "run.sh"), os.X_OK), "run.sh 必须可执行"
if shutil.which("docker"):
    d = subprocess.run(["docker", "compose", "-f", os.path.join(DEPLOY, "docker-compose.yml"),
                        "config", "--quiet"], capture_output=True, text=True, timeout=30)
    DOCKER_NOTE = ("守护进程可用且 compose 解析通过 ✔" if d.returncode == 0
                   else f"本机有 docker CLI 但守护进程不可达({(d.stderr or d.stdout).strip()[:90]})"
                        " → 课堂标注为『未执行』")
else:
    DOCKER_NOTE = "本机无 docker CLI → 容器部署未执行（不做假演示）"
print("docker 校验:", DOCKER_NOTE)

REPORT = os.path.join(REPORTS, "P7_A_选型与压测对比报告.md")
with open(REPORT, "w", encoding="utf-8") as f:
    f.write(f"""# 工单P7 · A层 选型 + 压测对比报告（本机实测，SEED={SEED}）

## 1. 技术决策矩阵（离线课堂可复现版）
| 场景 | 候选A | 候选B | 选择与理由（实测/断言留痕） |
|---|---|---|---|
| 设备台账 | SQLite(WAL) | MongoDB | 选 SQLite：journal_mode 实测={WAL_MODE}（SQLite {sqlite_version()}），读写不阻塞；台账量级小、要强一致外键 |
| 分钟级窗口特征 | 列式引擎(DuckDB) | SQLite逐行 | 选列式：全量{window_feats('PUMP-03', 120)['samples']}点窗口 列式{BENCH_WIN['columnar_ms']}ms vs 逐行{BENCH_WIN['rowwise_ms']}ms（{BENCH_WIN['speedup']}×） |
| 拓扑影响路径 | Kùzu 嵌入式 | Neo4j 服务 | 选嵌入式：零守护进程；影响面覆盖率断言=100%（{impact_path('PUMP-03')['downstream_count']}个下游设备） |
| 故障案例检索 | 向量库+嵌入 | 全文LIKE | 选向量+元数据过滤：recall@1 纯哈希{R_HASH['recall@1']} / 字符TF-IDF{R_TFIDF['recall@1']} / 哈希+设备过滤**{CHOSEN['recall@1']}**(选定)；bge-small-zh 需联网，本机未测不编数 |
| Web服务 | Litestar+Granian | http.server | 生产选前者；课堂用 http.server 同契约演示：IO等待型 多线程/单线程 = {BENCH_ROWS['gain_io']}×，CPU型 = {BENCH_ROWS['gain_cpu']}×（异步红利于IO等待） |

## 2. 压测明细（本机实测）
- 线上端点 /api/device/PUMP-03 并发梯度：{json.dumps(BENCH_ROWS['online'], ensure_ascii=False)}
- 并发模型对照(c=50, 各400请求)：
  IO等待3ms：单线程 {BENCH_ROWS['model']['IO等待3ms']['single']['qps']} → 多线程 {BENCH_ROWS['model']['IO等待3ms']['threaded']['qps']} QPS（{BENCH_ROWS['gain_io']}×）
  CPU忙等5ms：单线程 {BENCH_ROWS['model']['CPU忙等5ms']['single']['qps']} → 多线程 {BENCH_ROWS['model']['CPU忙等5ms']['threaded']['qps']} QPS（{BENCH_ROWS['gain_cpu']}×）
- 工单口径 ab -n 1000 -c 50：{json.dumps(BENCH_ROWS['ab'], ensure_ascii=False)}
- 结论：异步/Rust 多 workers 的收益主要在 **IO 等待重叠**；CPU 密集要真多进程或下沉到 C。
  同进程 stdlib 压测还会被客户端 GIL 卡住上限——压测工具也是被测系统的一部分。

## 3. 端到端验收
- verify()：{sum(1 for c in CHECKS if c[2])}/{len(CHECKS)} 全绿
- D层 4 API：全 200，live→cache 留痕见 reports/D_layer_live_vs_cache.json
- 容器部署：{DOCKER_NOTE}

## 4. 思政留痕
- 有据可查：智能体 {len(investigate('PUMP-03振动持续变大，会影响哪些产线？以前怎么处理的？')['trace'])} 步取证，每步含 thought/act/args/obs
- 全绿才上线：verify() 任一红灯即 assert 失败，禁止交付
""")
print("报告 →", os.path.relpath(REPORT, BASE),
      f"({os.path.getsize(REPORT)} 字节)")
assert os.path.getsize(REPORT) > 800, "A层报告内容不足"
print("✔ A层交付闸：三件套齐套 + compose 静态校验 + 报告含实测数字")


交付物 → w09_scratch/deploy | Dockerfile, docker-compose.yml, run.sh
compose 静态校验: 通过 ✔
docker 校验: 守护进程可用且 compose 解析通过 ✔
报告 → w09_scratch/reports/P7_A_选型与压测对比报告.md (2601 字节)
✔ A层交付闸：三件套齐套 + compose 静态校验 + 报告含实测数字


### Starter 23：部署入口 `deploy_entry()`——⚠ notebook 内核里 `sys.argv` **不是**模式来源
（教师版：下方为 **TODO 完成态**，可直接运行；学生版此处留空待填）

`run.sh` 用 `python deploy_entry.py seed|verify|serve|bench` 分发，靠的是 `__main__` + argv。
**在 notebook 内核里 `sys.argv` 装的是内核启动参数**（如 `-f /tmp/xxxx.json`），拿它判模式会**静默走错分支**——
本格先把这个坑打印出来，再用**显式入参** `deploy_entry("seed")` 调用。
这一格很短，但它是"脚本改 notebook"时最隐蔽的一次翻车，值得单独一格。

In [51]:
# -*- coding: utf-8 -*-
if __name__ == "__main__":
    # ⚠ notebook 内核里 sys.argv 装的是**内核启动参数**（如 -f /tmp/xxx.json），
    #   拿它判模式会静默走错分支——必须显式传参，argv 只作可选补充。
    known = [a for a in sys.argv[1:] if a in ("seed", "verify", "serve", "bench")]
    print("sys.argv[1:] =", sys.argv[1:], "→ 识别到的合法模式:", known or "无（改用显式入参）")


def deploy_entry(MODE="seed", port=0):
    """等价 starter 的 deploy/run.sh：seed(播种五库+模型) → verify(端到端) → serve(起服务)

    真实 CLI 用法（本 notebook 走同一函数，只是不阻塞）：
        /opt/anaconda3/envs/pp03/bin/python deploy_entry.py seed
        /opt/anaconda3/envs/pp03/bin/python deploy_entry.py verify
        granian --interface asgi service.app:app --host 127.0.0.1 --port 8900 --workers 4
    """
    if MODE == "seed":
        build_seed()
        seed_timeseries()
        seed_graph(); seed_vector(); train_and_save()
        print("五库 + 模型播种完成（幂等：drop→create→seed）")
    elif MODE == "verify":
        verify()
    elif MODE == "serve":
        srv, url = serve_background(port=port)
        print(f"服务已起 {url}（granian 上线时换 --workers 4）")
        return srv, url
    elif MODE == "bench":
        return bench("/health", concurrency=50, total=1000)
    return None


RUN_RESULT = deploy_entry("seed")            # notebook 里显式传参，绝不依赖 sys.argv
print("入口自检：seed 已复跑；verify/serve 在后续格按需触发")


sys.argv[1:] = ['W09_P7_智能运维平台综合_教师版.ipynb', 'W09_P7_智能运维平台综合_教师版_executed.ipynb'] → 识别到的合法模式: 无（改用显式入参）
[关系库] SQLite 3.50.2 | journal_mode=wal | 台账3台已播种
[时序库] 列存快照 21600 行 → vib_columnar.pkl | SQLite 原始表 21600 行(已建 (device,ts) 索引)
[图库] 节点11 关系12 → topology.json（Kùzu 离线替代：邻接表 + 可变跳遍历）
[向量库] 4 条案例 / 维度256 → fault_cases.npz（Chroma 离线替代：npz + numpy 余弦）
五步流 · 最优参数: {'max_depth': 6, 'n_estimators': 50}
特征重要度 [avg, max, slope, last, n/100]: [0.402 0.289 0.02  0.289 0.   ] → 斜率几乎没用(噪声骗人)，主力是均值/峰值/末值（呼应P2.2：斜率要平滑才可靠）
混淆矩阵
 [[26  3]
 [ 0 31]]
              precision    recall  f1-score   support

          正常      1.000     0.897     0.945        29
          磨损      0.912     1.000     0.954        31

    accuracy                          0.950        60
   macro avg      0.956     0.948     0.950        60
weighted avg      0.954     0.950     0.950        60

模型落盘 w09_scratch/data/models/p77_clf.joblib | F1(macro)=0.950 ≥ 0.80 ✔
五库 + 模型播种完成（幂等：drop→create→seed）
入口自检：seed 已复跑；verify/serve 在后

## 四、提交要求与避坑

**提交物**：`学号_姓名_P7`（本 notebook 含运行输出 / `p77_smart_ops` 目录）+ 分层工单（勾选层与达标自评）+ 过程记录 + 技术决策矩阵

**工单常见错误速查 → 本课对应实验**（每一条都在 notebook 里"演过一遍"）

| 工单里的现象 | 原因 | 解决 | 本课在哪验的 |
|---|---|---|---|
| granian 起不来 | 模块路径不对 | cd 到 `p77_smart_ops` 再启动 | Starter 22 `run.sh` 里 `cd "$(dirname "$0")/.."` |
| duckdb 报 STRUCT 转换错 | 把整个 dict 传给标量参数 | 绑定 `a["device"]` 再传 | Starter 5 注释：参数逐个绑定 |
| kuzu 重复主键 | CREATE 边时又建了同名节点 | 边要 MATCH 双端再 CREATE | Starter 7 播种用邻接表 + 显式去重 |
| chroma 首次卡住 | 默认嵌入要联网下载模型 | 用自带 HashEmbedding（离线） | Starter 9 自研哈希嵌入 |
| Litestar 422/400 | 处理器参数名/注解不符 | `request` 命名 + body 注解 dict | Starter 16 统一 400/404/500 兜底 |

**本课额外五条高频红字**
- ⚠ **五库要按顺序播种**（Starter 2/4/7/9/10），跳格运行 → `KeyError` / 文件找不到，别以为是 bug；
- ⚠ `window_feats` 的 **slope 分母用真实跨度**；`search` 的 **`dist = 1 - 相似度`** 别排反序；
- ⚠ ReAct 循环**必须把工具观察写回 `messages`**，否则永远重复同一工具撞满 8 步；
- ⚠ 压测一上量就 `connection reset` → 查 `request_queue_size`（默认 5）与 `protocol_version`，别急着怪 Python 慢；
- ⚠ **不许造数字**：决策矩阵与报告里的每个数字都必须来自本机打印；换机幅度会变、**趋势可复用**。

**工程边界**：本 notebook 只往 `Jupyter实训/w09_scratch/` 写，最后一格会把它整个删掉——
**课程包根目录零污染 + 顺序重跑结果一致（SEED=7）**，这两条本身就是要交付的东西。


### 💡 分层提示（先自己试5分钟再看）

**思路提示（每条对应一个 TODO 槽）**
> - **P7-1 台账**：`d = s.scalar(select(Device).where(Device.code == code))`；`if d is None: return {"found": False, ...}`；
>   否则顺 `d.points` / `d.alerts` 取从表——**必须在 `with Session` 块内访问**，出了块 session 已关，取到的是空。
> - **P7-2 窗口**：`cut = df["ts"].max() - timedelta(minutes=minutes)` → `w = df[df["ts"] >= cut]` → 五个聚合一次算完；
>   `slope = (w["vib_rms"].iloc[-1] - w["vib_rms"].iloc[0]) / 真实跨度分钟`（空数据返回全 0 形状，别抛异常）。
> - **P7-4 检索**：`np.load(VEC_NPZ)` → `q = embed_hash(query)` → `sims = vecs @ q`；
>   给了 `device` 就先按 `metas[i]["device"] == device` 缩候选再排序（不足 k 条回退全库补齐）；`dist = 1 - sims[i]`。
> - **P7-3 路径**：`follow(device_id, "上游供电", hops=(1, max_hops))` 拿变长跳下游 → 每个下游用 `lines_of(x)` **反查**产线
>   → `affected_lines` = 自己所在线 ∪ 波及线（去重 + 排序）。
> - **P7-5 循环**：`msg = chat(messages, TOOLS_SPEC)["choices"][0]["message"]`；`calls = msg.get("tool_calls") or []`；
>   `if not calls: break`；否则逐个 `run_tool(name, json.loads(call["function"]["arguments"]))`，
>   并把 **assistant 一条 + user 一条**都 append 回 `messages`，再 append trace。
> - **A 层**：三套检索方案用同一评测集量 recall@1；压测分清 **IO 等待 vs CPU 忙等**；报告三句结论各引用一个实测数字。

| 层 | 在本 notebook 里要做到什么 |
|---|---|
| **D 学徒** | 顺序跑到 Starter 17，看懂两组差异：`source: live → cache`、`cache: MISS → HIT`（这就是 4 个 API 的验收口径） |
| **C 操作员** | 补 P7-1 / P7-2 / P7-4，让 Starter 19 的**五库段** 5 行全 `[PASS]` |
| **B 组长** | 再补 P7-3 / P7-5，让 Starter 18 的 FAN-02 复测通过 + Starter 19 **ALL PASS（17/17）** |
| **A 师傅** | Starter 20/21/22 的实测数字抄进报告，写 `CONCLUSION` 三句选型（各引用倍数 / recall / QPS），并说清为什么 stdlib 压不出 granian 的 workers 红利 |

**报错自查三板斧**
① 读 `AssertionError` 括号里的**数据**（它直接告诉你缺哪一层，例如 `steps=1` = 观察没写回 messages）；
② 检查**顺序与状态**：五库是否都播种（跳格 → KeyError/文件不存在）、缓存是否被上一轮挡住（先 `KV.delete()`）；
③ 还红着就问同层同学 → 问教师；**不要用 `try/except` 把绿灯骗亮**（那是造假，答辩当场露馅）。

**自检**：D=`4个API全200` ｜ C=`五库段全绿` ｜ B=`verify ALL PASS` ｜ A=`报告三句结论都有实测数字`


## 教师参考答案（一键交付版 `p77()` · 对应 `教师参考/p77_smart_ops/`）

> **发放纪律：本附录与下方 `p77()` 单元勿随学生版下发。** 课堂按层放提示（D 层演示照做 / C 层只给思路行 / B 层点契约 / A 层直接做）。

`p77()` **不重写业务逻辑**，只做全流程组装 + D/C/B/A 四层逐级过闸：任一闸红灯立刻 `assert` 失败（= 全绿才上线），
最后再跑一次 `verify()` 并把 A 层报告全文打印出来，教师一条命令即可现场演示"从播种到交付"。

**学生版 TODO → 教师完成态 cell 对照表**（cell 号即 notebook 里的执行顺序，可直接跳转）

| TODO 编号 | 工单出处 | 层 | 学生版 cell | 教师版 cell | 关键实现 | 过闸断言位置 |
|---|---|---|---|---|---|---|
| （无槽·跟做） | D层 步骤1-3：run.sh + 四 API + live→cache 差异 | D | cell38 | cell38 | `run.sh` 文本落盘 + `http()` 五调 + 响应体逐字留痕 json | cell38 断言 / cell42 服务段 |
| **TODO-P7-1** | C层 步骤1 `device_profile` 关系查询 | C | cell10 | cell10 | `select` 主表 → relationship 取从表 → 告警倒序 → **有形状的空值** | cell10 末尾 / cell42 五库段 |
| **TODO-P7-2** | C层 步骤2 `window_feats` 时序窗口 | C | cell14 | cell14 | 布尔掩码截窗 + 列式聚合 + 真实跨度斜率；附「逐行」对照基准 | cell14 末尾 / cell42 五库段 |
| **TODO-P7-4** | C层 步骤3 `search` 向量检索 | C | cell22 | cell22 | npz 载入 → 嵌入查询 → **device 元数据过滤** → 余弦降序 → `dist=1-cos` | cell22 末尾 / cell42 五库段 |
| **TODO-P7-3** | B层 步骤1 `impact_path` 影响路径 | B | cell20 | cell20 | `follow(*1..3)` 变长跳 + `lines_of()` 反查产线 | cell20 覆盖率=100% / cell54 B层闸 |
| **TODO-P7-5** | B层 步骤2 `investigate` ReAct 循环 | B | cell34 | cell34 | 想→判→动→看四步 + trace 留痕 + **观察写回 messages** | cell34 五步全覆盖 / cell54 B层闸 |
| （B层 步骤3） | agent/ask 换 FAN-02 再测一轮 | B | cell40 | cell40 | 同一套代码：泵判『报警』、风机判『正常』 | cell40 / cell42 智能体段 |
| **TODO-P7-A** | A层：召回对比 / ab 压测 / compose 部署 | A | cell48(`CONCLUSION`) | cell48 + cell44 | 三方案 recall@1 实测 + IO/CPU 双基准 + 交付三件套 + 报告 | cell48 非空断言 / cell54 A层闸 |

> 表中 cell 号从 0 开始，与 notebook 左侧 `[n]:` 顺序一致。两版结构逐格对齐，仅学生版少了「本附录 + `p77()`」两格，故答案区编号略大于学生区。


In [55]:
# -*- coding: utf-8 -*-
def p77(show_report=True):
    """教师参考答案（一键交付版）：播种五库 → 四层逐级过闸 → 出厂总检

    与上方各 Starter 完成态互为校验：本格不重写业务逻辑，只做**全流程组装 +
    D/C/B/A 分层验收口径**。任一闸红灯立刻 assert 失败 = "全绿才上线"的工程纪律。
    对应 starter：`教师参考/p77_smart_ops/`（原厂五引擎版）；本 notebook 为离线白名单等价实现。
    """
    gates = []

    def gate(tier, name, metric, ok):
        gates.append((tier, name, metric, bool(ok)))
        assert ok, f"{tier}层闸门未过：{name} ({metric})"

    # ── D 层闸：平台跑通 = 4 个 API 全 200 + live→cache 差异 ──
    KV.delete("dev:PUMP-03"); KV.delete("pred:PUMP-03")
    codes = [http("GET", "/health")[0], http("GET", "/api/device/PUMP-03")[0],
             http("POST", "/api/predict", {"device": "PUMP-03"})[0],
             http("POST", "/api/agent/ask", {"question": "PUMP-03振动变大影响哪些产线？"})[0]]
    gate("D", "4个API全部200", f"status={codes}", codes == [200] * 4)
    KV.delete("dev:FAN-02")                                  # 先清空才能看到 live→cache
    src_seq = [http("GET", "/api/device/FAN-02")[1]["source"],
               http("GET", "/api/device/FAN-02")[1]["source"]]
    gate("D", "live→cache 可观察", "→".join(src_seq), src_seq == ["live", "cache"])

    # ── C 层闸：五库段全绿（关系/时序/图/向量/KV 各写读一次）──
    prof, wf = device_profile("PUMP-03"), window_feats("PUMP-03")
    gate("C", "关系台账查询", f"测点{prof['point_count']}/告警{prof['alert_count']}",
         prof["found"] and prof["point_count"] == 3)
    gate("C", "时序窗口特征", f"n={wf['samples']} slope={wf['slope']}", wf["samples"] > 300 and wf["slope"] > 0)
    gate("C", "向量案例检索", search("泵振动持续变大怎么办", 2, device="PUMP-03")[0]["id"],
         search("泵振动持续变大怎么办", 2, device="PUMP-03")[0]["id"] == "case-001")
    KV.put("p77:probe", "1", ttl=60)
    gate("C", "KV读写", f"ttl={KV.ttl_left('p77:probe')}", KV.get("p77:probe") == "1")

    # ── B 层闸：智能体 ALL PASS（图路径 + ReAct 五步 + 换设备复测）──
    ip = impact_path("PUMP-03")
    gate("B", "Cypher影响路径覆盖率100%", f"{ip['downstream_count']}个下游",
         {c["downstream"] for c in ip["collateral"]} == closure_reachable("PUMP-03", "上游供电"))
    for dev in ("PUMP-03", "FAN-02"):                      # 工单B层步骤3：问题换 FAN-02 再测一轮
        ag = investigate(f"{dev}振动趋势如何？会影响哪些产线？以前怎么处理？")
        exp_level = "报警" if dev == "PUMP-03" else "正常"
        gate("B", f"ReAct五步取证[{dev}]", f"steps={ag['steps']} level={ag['conclusion']['level']}",
             ag["steps"] == 5 and ag["conclusion"]["level"] == exp_level
             and set(ag["conclusion"]["evidence_keys"]) == set(REGISTRY)
             and all(t["thought"] and t["obs"] for t in ag["trace"]))

    # ── A 层闸：上线级指标（召回/延迟/QPS 全部实测）──
    gate("A", "案例召回@1 ≥ 0.80", f"A哈希{R_HASH['recall@1']} Btfidf{R_TFIDF['recall@1']} C过滤{CHOSEN['recall@1']}",
         CHOSEN["recall@1"] >= 0.80 and CHOSEN["recall@1"] > R_HASH["recall@1"])
    gate("A", "p95 延迟达标", f"{BENCH_ROWS['online'][2]['p95_ms']}ms(c50)",
         BENCH_ROWS["online"][2]["p95_ms"] < 500)
    gate("A", "并发红利IO>2×且CPU不涨", f"io={BENCH_ROWS['gain_io']}× cpu={BENCH_ROWS['gain_cpu']}×",
         BENCH_ROWS["gain_io"] > 2 and BENCH_ROWS["gain_cpu"] < BENCH_ROWS["gain_io"])
    gate("A", "交付三件套+报告齐套", f"{sorted(os.listdir(DEPLOY))}",
         {"run.sh", "Dockerfile", "docker-compose.yml"} <= set(os.listdir(DEPLOY))
         and os.path.getsize(REPORT) > 800)

    print(f"{'层':<3}{'闸门':<26}{'实测指标':<34}结论")
    for tier, name, metric, ok in gates:
        print(f"{tier:<4}{name:<28}{metric:<36}{'✔' if ok else '✗'}")
    chk = verify()
    if show_report:
        print("\n———— A层报告全文（reports/P7_A_选型与压测对比报告.md） ————")
        print(open(REPORT, encoding="utf-8").read())
    print(f"✔ 四层闸门 {len(gates)}/{len(gates)} 全绿 + 端到端 {sum(1 for c in chk if c[2])}/{len(chk)} 全绿 → 可交付")
    return {"gates": gates, "checks": chk}


P77 = p77()


层  闸门                        实测指标                              结论
D   4个API全部200                  status=[200, 200, 200, 200]         ✔
D   live→cache 可观察              live→cache                          ✔
C   关系台账查询                      测点3/告警2                             ✔
C   时序窗口特征                      n=600 slope=0.0217                  ✔
C   向量案例检索                      case-001                            ✔
C   KV读写                        ttl=60.0                            ✔
B   Cypher影响路径覆盖率100%           2个下游                                ✔
B   ReAct五步取证[PUMP-03]          steps=5 level=报警                    ✔
B   ReAct五步取证[FAN-02]           steps=5 level=正常                    ✔
A   案例召回@1 ≥ 0.80               A哈希0.833 Btfidf0.833 C过滤1.0         ✔
A   p95 延迟达标                    29.24ms(c50)                        ✔
A   并发红利IO>2×且CPU不涨             io=9.38× cpu=0.99×                  ✔
A   交付三件套+报告齐套                  ['Dockerfile', 'docker-compose.yml', 'run.sh']✔
[PASS] 五库   关系

## 运行与验证

**教师版运行方式**：顺序 Shift+Enter 到底，约 20 秒，`p77()` 那一格就是"一键交付演示"。
- 首格打印「引擎体检 + 版本」，末格停服务并 `shutil.rmtree(w09_scratch)` —— **课程包根目录不留任何产物**；
- 服务用 `port=0` 随机端口，**不占课堂的 8900**，也不会阻塞内核；
- 结果可复现：`SEED=7` + `RandomState` + 假时钟，全班每次看到同一组数字（压测 QPS 除外，机器相关，格内已标注）。

**验收对照（本机实测，见 executed 版对应 cell 的真实输出）**

| 闸门 | 工单口径 | 本 notebook 的落点 |
|---|---|---|
| D | 4 个 API 全 200 | Starter 17：`statuses == [200]*5` + live→cache 留痕 json |
| C | verify 五库段全绿 | Starter 19：五库段 5 行 `[PASS]` |
| B | verify ALL PASS | Starter 18 FAN-02 复测 + Starter 19 `17/17 checks green` |
| A | 选型 + 压测对比报告 | Starter 20/21/22：三方案 recall@1 + IO/CPU 双基准 + 报告全文打印 |

> 配套：学生版 `W09_P7_智能运维平台综合_学生版.ipynb`（5 个 TODO 槽 + A 层 `CONCLUSION`，脚手架完整、答案留空）。
> 原厂实现见 `学生代码框架/p77_smart_ops/`（真实 Litestar/DuckDB/Kùzu/Chroma/Redis 版）与 `教师参考/p77_smart_ops/`；
> 本 notebook 是它的**离线等价教学版**，接口契约一致，可逐格对照讲解。


In [57]:
# -*- coding: utf-8 -*-
# --- 收尾：停服务 + 清理实验目录（课程包根目录全程零污染） ---
try:
    SRV.shutdown()
    SRV.server_close()
    print("[服务] 已优雅停机")
except Exception as e:
    print("[服务] 停机异常(可忽略):", type(e).__name__, e)

print("清理前 scratch 内容:", sorted(os.listdir(SCRATCH)))
shutil.rmtree(SCRATCH, ignore_errors=True)
print("清理后:", "w09_scratch 已删除" if not os.path.exists(SCRATCH) else "仍在",
      "| 课程包根目录未被写入任何运行产物")
print("复现说明：从头顺序运行本 notebook 即可重建全部产物（SEED=%d 结果一致）" % SEED)


[服务] 已优雅停机
清理前 scratch 内容: ['data', 'deploy', 'reports']
清理后: w09_scratch 已删除 | 课程包根目录未被写入任何运行产物
复现说明：从头顺序运行本 notebook 即可重建全部产物（SEED=7 结果一致）
